# 1. Information about the submission

## 1.1 Name and number of the assignment

Group Assignment

Hallucination Detection in Tool Calling

## 1.2 Students name

Svetlana Lukina, Hassan Iftikhar, Ali Arshad, Veronika Morozova, Ilya Sharov

## 1.3 Workflow

![Hallucination detection workflow](workflow.png)


## 1.3 Dataset Hugging Face link

https://huggingface.co/datasets/Ali-Bhai/Hallucination_Dataset/

## 2. Data & Methodology

We study **span-level hallucination detection** in tool-augmented assistant answers. Each example is a tuple `(query, context, output)` where `context` is the tool output and `output` is the final assistant answer. The task is to predict which character spans in `output` are unsupported or inconsistent with the available tool evidence.

**Pipeline (ToolACE → RAGTruth-style labels):**

1. Extract tool-use episodes from ToolACE dialogues.
2. Keep one **clean** answer per episode.
3. Inject synthetic corruptions into the clean answer.
4. Annotate gold hallucination spans.
5. Split by `source_id` into train / validation / test.
6. Train and evaluate detectors with **micro character-level P/R/F1** (same metric for all methods).


### Task vs. classic RAGTruth

| Classic RAGTruth | Our ToolACE setup |
|---|---|
| Document → summary | Tool output → assistant answer |
| Hallucination vs. source document | Hallucination vs. tool output + tool metadata |
| Span labels in `hallucination_labels` | Same |

### Field mapping (ToolACE → RAGTruth)

| Concept | ToolACE source | Our field |
|---|---|---|
| User query | `conversations` (`from=="user"`) | `query` |
| Available tools | JSON in `system` | `available_tools`, `tool_names` |
| Tool call | assistant turn before `tool` | `tool_call` |
| Tool output | `tool` turn | `context` |
| Final answer | assistant turn after `tool` | `output` |
| Gold spans | injected in corruption stage | `hallucination_labels` |
| Clean reference | original episode answer | `clean_output` (evaluation / QC only) |

### How hallucination data were created

Starting from **clean** ToolACE episodes, we generated three corruption types on the assistant answer:

- **Contradiction** — small factual edits that conflict with tool output (numbers, polarity, entities).
- **Missing tool** — claims of actions never supported by tools (email sent, reminder scheduled, etc.).
- **Overgeneration** — extra unsupported content added to an otherwise valid answer.

Each corrupted row stores character spans in `hallucination_labels`. The corrupted `output` is the **same answer as clean with a small edited region** (append or local change) — not a full regeneration. The detector never receives `clean_output` or gold spans at **inference**; `clean_output` is used only for dataset QC and evaluation.

### Dataset overview (current run)

| Item | Count | Notes |
|---|---:|---|
| Contradiction hallucinated rows | 1291 | `toolace_contradiction_ragtruth.jsonl` |
| Overgeneration hallucinated rows | 1347 | `toolace_overgeneration_ragtruth.jsonl` |
| Missing-tool hallucinated rows | 1178 | `toolace_missing_tool_ragtruth.jsonl` |
| Unique clean rows | 1347 | one per episode |
| Total canonical hallucinated | 3816 | sum of the three corruption files |
| Total mixed rows | 5163 | hallucinated + unique clean |

### Episode-level split (`source_id`, seed=42)

Split is **70% / 10% / 20%** of unique episodes (not 80/10/10). All variants of the same episode stay in one split.

| Split | Unique episodes | Mixed rows | Clean | Hallucinated |
|---|---:|---:|---:|---:|
| Train | 942 | 3617 | 942 | 2675 |
| Validation | 134 | 514 | 134 | 380 |
| Test | 271 | 1032 | 271 | 761 |

### `mixed_test` breakdown

| Subset | Rows |
|---|---:|
| Contradiction (corrupted only) | 257 |
| Missing-tool (corrupted only) | 233 |
| Overgeneration (corrupted only) | 271 |
| Clean test rows | 271 |
| **Mixed test total** | **1032** |

Per-type evaluation uses: corrupted rows of that type **+ all test clean rows** (same predictions, no re-inference).

### Dataset artifact diagnostics

Checks whether simple heuristics explain our synthetic labels. Gold spans are **not** always the last sentence; contradiction spans are often short and mid-answer.

**Gold span position stats (test corrupted subsets):**

| Type | n | Ends at answer end | Exact last sentence | Subset of last sentence |
|---|---:|---:|---:|---:|
| contradiction | 257 | 1.2% | 0.0% | 18.7% |
| missing_tool | 233 | 28.3% | 26.6% | 40.8% |
| overgeneration | 271 | 2.6% | 0.4% | 34.3% |

**Heuristic baselines (micro char P/R/F1):**

| Split | Heuristic | P | R | F1 |
|---|---|---:|---:|---:|
| contradiction (n=257) | last sentence | 0.029 | 0.160 | 0.049 |
| contradiction (n=257) | missing-tool regex | 0.000 | 0.000 | 0.000 |
| missing_tool (n=233) | last sentence | 0.302 | 0.405 | 0.346 |
| missing_tool (n=233) | missing-tool regex | 0.361 | 0.031 | 0.058 |
| overgeneration (n=271) | last sentence | 0.160 | 0.352 | 0.219 |
| overgeneration (n=271) | missing-tool regex | 0.000 | 0.000 | 0.000 |
| mixed_test (n=1032) | last sentence | 0.119 | 0.352 | 0.178 |
| mixed_test (n=1032) | missing-tool regex | 0.361 | 0.015 | 0.028 |

**Takeaway:** naive positional heuristics fail → real detectors must use tool context. Label–diff QC is high for missing-tool / overgeneration (localized injections match annotations). Contradiction has lower recall because gold spans can mark a semantic flip slightly broader than the minimal diff.

### Improved method: tool-aware DeBERTa-v3-base

Our main improvement fine-tunes **`microsoft/deberta-v3-base`** as a **token classifier** (Lettuce-style span detection, but trained on our data).

**Input at inference:** `query`, available tools, tool names, tool call, tool output (`context`), and final answer (`output`).  
**Not used at inference:** `clean_output`, gold spans.

**Training:** token labels from gold spans → char scores → predicted spans; **8 epochs**; **contradiction oversampling (×3)** on train; checkpoint selected by **validation char F1**; threshold tuned on `mixed_val`.

## 2.2 Discussion of results

All scores: **micro character-level P/R/F1** on `mixed_test` (1032 rows = 761 corrupted + 271 clean).

### Overall comparison (`mixed_test`)

| Method | P | R | F1 |
|---|---:|---:|---:|
| LettuceDetect (zero-shot) | 0.239 | 0.373 | 0.291 |
| LookBackLens (TinyLlama + attention head) | 0.294 | 0.901 | 0.443 |
| Tool-aware DeBERTa (ours) | 0.961 | 0.929 | **0.945** |
| Heuristic: last sentence | 0.119 | 0.352 | 0.178 |
*(Dataset QC — label–diff vs clean, offline only: F1 0.93; not a detector.)*

DeBERTa is strongest overall because it is **supervised** on our span labels with full tool-aware context. LookBack has high recall but lower precision (wide spans). Lettuce is zero-shot and struggles on this domain.

### Per corruption type (corrupted + all test clean)

| Type | Lettuce F1 | LookBack F1 | DeBERTa F1 |
|---|---:|---:|---:|
| contradiction | 0.088 | 0.092 | **0.651** |
| missing_tool | 0.290 | 0.462 | **0.978** |
| overgeneration | 0.269 | 0.395 | **0.956** |

**Contradiction** remains hardest for all methods, but DeBERTa improves most (0.09 → 0.65 per-type slice) thanks to oversampling and validation-F1 checkpoint selection. **Missing-tool** and **overgeneration** are easier (explicit unsupported actions or added claims).

### Comparative examples: DeBERTa vs LookBack vs LettuceDetect

**Clean output** and **Answer** are for comparison only (not model inputs). Each row shows **gold spans** and predictions from all three detectors on the **same** test example.


---

#### `508_ep1_contradiction` — contradiction

**Clean output** (reference only):

> The valve timings have been successfully set and confirmed for your upcoming batch of **300 liters**, with operations scheduled at 08:00, 12:00, and 16:00, each lasting 20 minutes.

**Answer (output):**

> The valve timings have been successfully set and confirmed for your upcoming batch of **250 liters**, with operations scheduled at 08:00, 12:00, and 16:00, each lasting 20 minutes.

| | Spans |
|---|---|
| **Gold** | `[86:96]` *250 liters* |
| **DeBERTa predicted** | `[86:96]` *250 liters* |
| **LookBack predicted** | `[68:70]` *up* · `[86:130]` *250 liters, with operations scheduled at 08:* |
| **Lettuce predicted** | *none* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 1.000 | 1.000 | 1.000 |
| LookBack | 0.217 | 1.000 | 0.357 |
| Lettuce | 0.000 | 0.000 | 0.000 |

*Lettuce predicts nothing; DeBERTa exact, LookBack wide.*

---

#### `442_ep1_contradiction` — contradiction

**Clean output** (reference only):

> It seems that the vitals data from the patient monitoring device **has been retrieved**, but no specific details are displayed. If you need help interpreting this data or require specific types of vitals, please let me know how else I can assist you!

**Answer (output):**

> It seems that the vitals data from the patient monitoring device **has not been retrieved**, but no specific details are displayed. If you need help interpreting this data or require specific types of vitals, please let me know how else I can assist you!

| | Spans |
|---|---|
| **Gold** | `[65:87]` *has not been retrieved* |
| **DeBERTa predicted** | `[65:87]` *has not been retrieved* · `[93:104]` *no specific* · `[113:126]` *are displayed* |
| **LookBack predicted** | `[69:130]` *not been retrieved, but no specific details are displayed. If* · `[233:250]` *I can assist you!* |
| **Lettuce predicted** | `[73:87]` *been retrieved* · `[89:130]` *but no specific details are displayed. If* · `[163:167]` *data* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 0.478 | 1.000 | 0.647 |
| LookBack | 0.231 | 0.818 | 0.360 |
| Lettuce | 0.636 | 0.237 | 0.346 |

*Lettuce flags wrong polarity fragment plus tail noise.*

---

#### `392_ep0_contradiction` — contradiction

**Clean output** (reference only):

> The carbon footprint for your car travel from San Francisco to Los Angeles using a typical gas vehicle is approximately **145.84** kg CO2e.

**Answer (output):**

> The carbon footprint for your car travel from San Francisco to Los Angeles using a typical gas vehicle is approximately **245.84** kg CO2e.

| | Spans |
|---|---|
| **Gold** | `[120:126]` *245.84* |
| **DeBERTa predicted** | `[120:129]` *245.84 kg* |
| **LookBack predicted** | `[120:133]` *245.84 kg CO2* |
| **Lettuce predicted** | *none* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 0.667 | 1.000 | 0.800 |
| LookBack | 0.462 | 1.000 | 0.632 |
| Lettuce | 0.000 | 0.000 | 0.000 |

*Lettuce misses the changed number.*

---

#### `600_ep1_missing_tool` — missing_tool

**Clean output** (reference only):

> Based on the issues found, here are some resources to help you address these accessibility issues related to WCAG2.1:
> 1. **Images missing alt text:** [Guidelines for providing text alternatives for non-text content](https://www.w3.org/WAI/WCAG21/quickref/#text-alternatives)
> 2. **Insufficient color contrast:** [Guidelines for ensuring sufficient color contrast](https://www.w3.org/WAI/WCAG21/quickref/#contrast-minimum)
>
> These resources should provide detailed information and steps to make your website compliant with WCAG2.1 standards.

**Answer (output):**

> Based on the issues found, here are some resources to help you address these accessibility issues related to WCAG2.1:
> 1. **Images missing alt text:** [Guidelines for providing text alternatives for non-text content](https://www.w3.org/WAI/WCAG21/quickref/#text-alternatives)
> 2. **Insufficient color contrast:** [Guidelines for ensuring sufficient color contrast](https://www.w3.org/WAI/WCAG21/quickref/#contrast-minimum)
>
> These resources should provide detailed information and steps to make your website compliant with WCAG2.1 standards**, I have also scheduled a reminder for you to review these guidelines next week.**

| | Spans |
|---|---|
| **Gold** | `[537:616]` *, I have also scheduled a reminder for you to review these guidelines next week* |
| **DeBERTa predicted** | `[537:616]` *(exact match)* |
| **LookBack predicted** | `[537:617]` *, I have also scheduled a reminder for you to review these guidelines next week.* |
| **Lettuce predicted** | *none* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 1.000 | 1.000 | 1.000 |
| LookBack | 0.975 | 1.000 | 0.988 |
| Lettuce | 0.000 | 0.000 | 0.000 |

*Lettuce misses the appended reminder sentence.*

---

#### `94_ep0_missing_tool` — missing_tool

**Clean output** (reference only):

> The average gas price in Toronto, Ontario is $1.25 per liter.

**Answer (output):**

> The average gas price in Toronto, Ontario is $1.25 per liter. **I've sent this price update to your email for your records.**

| | Spans |
|---|---|
| **Gold** | `[62:121]` *I've sent this price update to your email for your records.* |
| **DeBERTa predicted** | `[62:121]` *(exact match)* |
| **LookBack predicted** | `[49:121]` *5 per liter. I've sent this price update to your email for your records.* |
| **Lettuce predicted** | `[0:121]` *(entire answer including factual price line)* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 1.000 | 1.000 | 1.000 |
| LookBack | 0.819 | 1.000 | 0.901 |
| Lettuce | 1.000 | 0.488 | 0.656 |

*Lettuce recalls gold but marks the whole answer.*

---

#### `17_ep1_missing_tool` — missing_tool

**Clean output** (reference only):

> The current price of the CAKE token on PancakeSwap is $4.25.

**Answer (output):**

> The current price of the CAKE token on PancakeSwap is $4.25. **I have scheduled a reminder to notify you if the price changes significantly.**

| | Spans |
|---|---|
| **Gold** | `[61:138]` *I have scheduled a reminder to notify you if the price changes significantly.* |
| **DeBERTa predicted** | `[61:138]` *(exact match)* |
| **LookBack predicted** | `[61:138]` *(exact match)* |
| **Lettuce predicted** | `[54:59]` *$4.25* · `[61:138]` *I have scheduled a reminder to notify you if the price changes significantly.* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 1.000 | 1.000 | 1.000 |
| LookBack | 1.000 | 1.000 | 1.000 |
| Lettuce | 1.000 | 0.939 | 0.969 |

*Lettuce hits gold; also flags the factual price.*

---

#### `171_ep1_overgeneration` — overgeneration

**Clean output** (reference only):

> The name "Riley" is mostly associated with the female gender, with a probability of 0.58 or 58%. However, names can be used for any gender, and the result is based on common usage observed from the dataset used by the tool.

**Answer (output):**

> The name "Riley" is mostly associated with the female gender, with a probability of 0.58 or 58%, **and it is the most popular girl's name in several countries**. However, names can be used for any gender, and the result is based on common usage observed from the dataset used by the tool.

| | Spans |
|---|---|
| **Gold** | `[97:156]` *and it is the most popular girl's name in several countries* |
| **DeBERTa predicted** | `[97:156]` *(exact match)* |
| **LookBack predicted** | `[69:80]` *probability* · `[92:179]` *58%, and it is the most popular girl's name in several countries. However, names can be* · `[199:234]` *, and the result is based on common* |
| **Lettuce predicted** | *none* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 1.000 | 1.000 | 1.000 |
| LookBack | 0.444 | 1.000 | 0.615 |
| Lettuce | 0.000 | 0.000 | 0.000 |

*Lettuce misses the inserted claim; DeBERTa/LookBack find it.*

---

#### `28_ep1_overgeneration` — overgeneration

**Clean output** (reference only):

> I'm here to help! Here's a summary of the article about Venice's architectural treasures:
>
> ### Architectural Exhibition in Venice: Living with Water
>
> In 2026, Venice hosted an architectural exhibition focused on **sustainable living**. … Notable proposals included:
>
> - **Floating homes**
> - **Buildings with bio-reactive facades**
> - **Structures that integrate with the aquatic ecosystem**
>
> This exhibition highlighted the city's commitment to future-proofing its architecture against environmental challenges.
>
> If you need more details or have any other questions, feel free to ask!

**Answer (output):**

> *(same as clean, except the closing paragraph adds:)* **, drawing visitors from around the world**

| | Spans |
|---|---|
| **Gold** | `[653:691]` *drawing visitors from around the world* |
| **DeBERTa predicted** | *none* |
| **LookBack predicted** | `[653:692]` *drawing visitors from around the world.* · `[114:129]` *ition in Venice* |
| **Lettuce predicted** | `[533:582]` *This exhibition highlighted the city's commitment* · `[653:691]` *drawing visitors from around the world* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 0.000 | 0.000 | 0.000 |
| LookBack | 0.704 | 1.000 | 0.826 |
| Lettuce | 1.000 | 0.437 | 0.608 |

*Lettuce finds gold plus extra context before it.*

---

#### `448_ep0_overgeneration` — overgeneration

**Clean output** (reference only):

> The box office information for "The Dark Knight" (ID: tt0468569) is as follows:
>
> - **Worldwide Gross Revenue:** $1,004,558,444
> - **Domestic Gross Revenue (US):** $533,345,358
> - **International Gross Revenue:** $470,212,686

**Answer (output):**

> The box office information for "The Dark Knight" (ID: tt0468569) is as follows:
>
> - **Worldwide Gross Revenue:** $1,004,558,444**, which made it the highest-grossing film of 2008**
> - **Domestic Gross Revenue (US):** $533,345,358
> - **International Gross Revenue:** $470,212,686

| | Spans |
|---|---|
| **Gold** | `[128:175]` *which made it the highest-grossing film of 2008* |
| **DeBERTa predicted** | `[128:175]` *(exact match)* |
| **LookBack predicted** | `[68:79]` *as follows:* · `[114:162]` *,004,558,444, which made it the highest-grossing* · `[219:221]` *,3* · `[222:223]` *8* |
| **Lettuce predicted** | `[83:85]` *** · `[90:175]` *wide Gross Revenue:** $1,004,558,444, which made it the highest-grossing film of 2008* |

| Model | P | R | F1 |
|---|---:|---:|---:|
| DeBERTa | 1.000 | 1.000 | 1.000 |
| LookBack | 0.548 | 0.723 | 0.624 |
| Lettuce | 1.000 | 0.540 | 0.701 |



# Code

In [2]:
# Core dependencies for the hallucination-detection pipeline
%pip install -q datasets huggingface_hub pandas pyarrow tqdm

Note: you may need to restart the kernel to use updated packages.


## ToolACE → RAGTruth-style setup

We convert ToolACE tool-use dialogues into span-labeled examples:

```
ToolACE episodes
  → extract (query, tool_call, tool_output, clean answer)
  → inject contradiction / missing_tool / overgeneration
  → annotate character spans (hallucination_labels)
  → episode-level train / val / test split
  → train & evaluate span detectors
```

**Detection target:** unsupported text in the **assistant answer**, given the **tool output** and tool metadata — not document-to-summary hallucination as in classic RAGTruth.


In [1]:
from datasets import load_dataset

# ToolACE: synthetic multi-turn tool-calling dialogues (11.3k train rows)
# https://huggingface.co/datasets/Team-ACE/ToolACE
toolace = load_dataset("Team-ACE/ToolACE", split="train") #not for hallucination detection

# RAGTruth-processed: reference format for span-level hallucination labels
# https://huggingface.co/datasets/wandb/RAGTruth-processed
ragtruth = load_dataset("wandb/RAGTruth-processed")

print("ToolACE:", toolace)
print("RAGTruth:", ragtruth)
print("ToolACE columns:", toolace.column_names)
print("RAGTruth columns:", ragtruth["train"].column_names)

Repo card metadata block was not found. Setting CardData to empty.


ToolACE: Dataset({
    features: ['system', 'conversations'],
    num_rows: 11300
})
RAGTruth: DatasetDict({
    train: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 15090
    })
    test: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 2700
    })
})
ToolACE columns: ['system', 'conversations']
RAGTruth columns: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str']


ToolACE:
'context': tool outputs, retrieved docs, tool traces

RAGTruth:
'output' contains hallucinations
'hallucination_labels':[
  {
    "start": 35,
    "end": 72,
    "label": "hallucination"
  }
]
'input': {
  question: ...
  passages: ...
  output: ...
}

In [2]:
import json
import re
from collections import Counter

# --- ToolACE top-level schema ---
print("=== ToolACE schema ===")
print(toolace.features)
print("\nExample keys:", list(toolace[0].keys()))

# `system` contains instructions + JSON tool definitions
print("\n--- system (first 600 chars) ---")
print(toolace[0]["system"][:600])

# `conversations` is a list of turns: user | assistant | tool
print("\n--- conversations (turn roles) ---")
for i, turn in enumerate(toolace[0]["conversations"]):
    preview = turn["value"].replace("\n", " ")[:120]
    print(f"  [{i}] from={turn['from']!r:10s} len={len(turn['value']):5d}  {preview}...")

=== ToolACE schema ===
{'system': Value('string'), 'conversations': List({'from': Value('string'), 'value': Value('string')})}

Example keys: ['system', 'conversations']

--- system (first 600 chars) ---
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:
[{"name": "newAddress", "description": "Generates a new Ethereum address that can be used to send or receive funds. Do not lose t

--- conversations (turn roles) ---
  [0] from='user'     len=  138  I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market...
  

### ToolACE turn semantics

Each row is one **multi-turn dialogue**, not one tool call.

Typical **tool-use cycle** inside `conversations`:

```
user      → question
assistant → tool call as string, e.g. [Weather_API(location="Beijing")]
tool      → JSON list with API name + results
assistant → natural-language answer grounded in tool output
```

Later turns may repeat the cycle or continue without tools (general chat).

In [3]:
def parse_tools_from_system(system: str) -> list[dict]:
    """Extract the JSON tool list embedded in the system prompt.

    ToolACE appends formatting instructions after the JSON array, e.g.
    ``].\\nShould you decide to return the function call(s)...``
    so we parse only the first JSON value (raw_decode), not the whole tail.
    """
    marker = "Here is a list of functions in JSON format that you can invoke:\n"
    if marker not in system:
        return []
    payload = system.split(marker, 1)[1].strip()
    tools, _end = json.JSONDecoder().raw_decode(payload)
    if not isinstance(tools, list):
        raise ValueError(f"Expected a JSON list of tools, got {type(tools)}")
    return tools


def is_tool_call(text: str) -> bool:
    """Heuristic: assistant turn that looks like [ApiName(...)], not tool JSON."""
    t = text.strip()
    if not (t.startswith("[") and "]" in t):
        return False
    # Tool role payloads are JSON lists, not bracket calls
    if t.startswith("[{") or t.startswith("[\n{"):
        return False
    return True


def extract_tool_episodes(conversations: list[dict]) -> list[dict]:
    """Extract one episode per `tool` turn (supports multi-tool dialogues).

    For each tool message, walk backward to the nearest tool-call assistant
    and user, then forward to the first natural-language assistant reply.
    """
    episodes = []
    n = len(conversations)

    for ti in range(n):
        if conversations[ti]["from"] != "tool":
            continue

        call_idx = None
        for j in range(ti - 1, -1, -1):
            if conversations[j]["from"] == "assistant":
                if is_tool_call(conversations[j]["value"]):
                    call_idx = j
                break
        if call_idx is None:
            for j in range(ti - 1, -1, -1):
                if conversations[j]["from"] == "assistant" and is_tool_call(
                    conversations[j]["value"]
                ):
                    call_idx = j
                    break
        if call_idx is None:
            continue

        user_idx = None
        for j in range(call_idx - 1, -1, -1):
            if conversations[j]["from"] == "user":
                user_idx = j
                break
        if user_idx is None:
            continue

        ans_idx = None
        for j in range(ti + 1, n):
            if conversations[j]["from"] == "assistant" and not is_tool_call(
                conversations[j]["value"]
            ):
                ans_idx = j
                break
        if ans_idx is None:
            continue

        episodes.append(
            {
                "user_query": conversations[user_idx]["value"],
                "tool_call": conversations[call_idx]["value"],
                "tool_output": conversations[ti]["value"],
                "assistant_answer": conversations[ans_idx]["value"],
                "turn_indices": {
                    "user": user_idx,
                    "call": call_idx,
                    "tool": ti,
                    "answer": ans_idx,
                },
            }
        )
    return episodes


# Demo on one ToolACE row
row = toolace[0]
tools = parse_tools_from_system(row["system"])
episodes = extract_tool_episodes(row["conversations"])

print(f"Available tools in system prompt: {len(tools)}")
print(f"Tool names (first 5): {[t['name'] for t in tools[:5]]}")
print(f"Extracted tool-use episodes in this dialogue: {len(episodes)}")

if episodes:
    ep = episodes[0]
    for key in ep:
        print(f"\n--- {key} ---")
        val = ep[key]
        text = val if isinstance(val, str) else repr(val)
        print(text[:500])

Available tools in system prompt: 6
Tool names (first 5): ['newAddress', 'Market Trends API', 'Get Futures Prices', 'SEC Filings', 'United States Away from Home Mobility API']
Extracted tool-use episodes in this dialogue: 1

--- user_query ---
I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?

--- tool_call ---
[Market Trends API(trend_type="MARKET_INDEXES", country="us")]

--- tool_output ---
[{"name": "Market Trends API", "results": {"trends": [{"name": "S&P 500", "description": "Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J", "description": "Dow Jones Industrial Average is a price-weighted average of 30 blue-chip stocks that are generally the leaders in their industry.", "data": {"current_value": "34479.60", "percentage_c

--- assistan

### RAGTruth-processed format (target for our benchmark)

Fields we must reproduce when exporting corrupted ToolACE data:

- `query` — user question
- `context` — retrieved / tool evidence (for us: **tool output**)
- `output` — model answer (we will corrupt this in Stage 1)
- `hallucination_labels` — JSON list of character spans `{start, end, text, label_type, ...}`

For baselines (LettuceDetect / LookBackLens): **question** = `query`, **context** = tool output, **answer** = `output`.

In [4]:
# --- RAGTruth schema ---
print("=== RAGTruth schema (train) ===")
print(ragtruth["train"].features)

# Find an example with non-empty span labels
labeled_idx = None
for idx in range(min(5000, len(ragtruth["train"]))):
    labels = ragtruth["train"][idx]["hallucination_labels"]
    if labels and labels not in ("[]", []):
        labeled_idx = idx
        break

print("First labeled example index:", labeled_idx)
ex = ragtruth["train"][labeled_idx]
print("\nquery:", ex["query"][:200])
print("\ncontext:", ex["context"][:200], "...")
print("\noutput:", ex["output"][:300], "...")
print("\nhallucination_labels (raw JSON string):")
print(ex["hallucination_labels"][:800])
print("\nhallucination_labels_processed:", ex["hallucination_labels_processed"])

# Parse span annotations (character offsets in `output`)
labels = json.loads(ex["hallucination_labels"]) if isinstance(ex["hallucination_labels"], str) else ex["hallucination_labels"]
for span in labels[:2]:
    snippet = ex["output"][span["start"] : span["end"]]
    print(f"\nSpan [{span['start']}:{span['end']}] type={span.get('label_type')}")
    print("  text:", snippet)
    print("  meta:", span.get("meta", "")[:120])

=== RAGTruth schema (train) ===
{'id': Value('string'), 'query': Value('string'), 'context': Value('string'), 'output': Value('string'), 'task_type': Value('string'), 'quality': Value('string'), 'model': Value('string'), 'temperature': Value('float32'), 'hallucination_labels': Value('string'), 'hallucination_labels_processed': {'evident_conflict': Value('int32'), 'baseless_info': Value('int32')}, 'input_str': Value('string')}
First labeled example index: 2

query: Summarize the following news within 116 words:

context: Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she ha ...

output: New research conducted by the Anne Frank House has revealed that Anne Frank and her sister Margot likely died in the Bergen-Belsen concentration camp at least a month earlier than previously believed. The researchers examined archives of the Red Cross, the Int

query   = user_query 

context = tool_output (+tool_call)

output  = assistant_answer

In [5]:
def toolace_episode_to_ragtruth_row(
    episode: dict,
    tools: list[dict],
    dialogue_id: str,
    episode_idx: int,
) -> dict:
    """Map one ToolACE tool-use episode to RAGTruth-style fields (clean, pre-corruption)."""
    return {
        "id": f"{dialogue_id}_ep{episode_idx}",
        "query": episode["user_query"],
        "context": episode["tool_output"],  # tool evidence (assignment: tool response)
        "output": episode["assistant_answer"],  # clean answer; corrupt in Stage 1
        "tool_call": episode["tool_call"],
        "available_tools": json.dumps([t["name"] for t in tools]),
        "hallucination_labels": "[]",  # filled after synthetic corruption
    }


# Flatten a small sample to estimate how many evaluation rows we get
SAMPLE_N = 200
flat_rows = []
role_counts = Counter()
episode_counts = []

for idx in range(SAMPLE_N):
    row = toolace[idx]
    tools = parse_tools_from_system(row["system"])
    episodes = extract_tool_episodes(row["conversations"])
    episode_counts.append(len(episodes))
    for role in row["conversations"]:
        role_counts[role["from"]] += 1
    for ep_i, ep in enumerate(episodes):
        flat_rows.append(toolace_episode_to_ragtruth_row(ep, tools, str(idx), ep_i))

print(f"Dialogues sampled: {SAMPLE_N}")
print(f"Role counts (all turns): {dict(role_counts)}")
print(f"Episodes per dialogue — mean: {sum(episode_counts)/len(episode_counts):.2f}, max: {max(episode_counts)}")
print(f"Flattened tool-use rows: {len(flat_rows)} (~{len(flat_rows)/SAMPLE_N:.2f} per dialogue)")

print("\n--- Example flattened row (ready for Stage 1 corruption) ---")
for k, v in flat_rows[0].items():
    print(f"{k}: {str(v)[:300]}")

Dialogues sampled: 200
Role counts (all turns): {'user': 663, 'assistant': 963, 'tool': 300}
Episodes per dialogue — mean: 1.50, max: 3
Flattened tool-use rows: 299 (~1.50 per dialogue)

--- Example flattened row (ready for Stage 1 corruption) ---
id: 0_ep0
query: I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?
context: [{"name": "Market Trends API", "results": {"trends": [{"name": "S&P 500", "description": "Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J",
output: Here are the top Market Trends in the US right now:

1. **S&P 500**: The Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies. Its current value is 4172.80 with a percentage change of +0.68%.

2. **DOW J**: The 

###ToolACE → hallucination dataset

1. **ToolACE storage:** each row = `system` + `conversations[]` with `from ∈ {user, assistant, tool}`.
2. **Tools** live in `system` as a JSON array (name, description, parameters).
3. **One dialogue → many tool episodes**; flatten to one RAGTruth row per `(user → tool_call → tool → answer)` cycle.
4. **RAGTruth labels** use **character offsets** in `output`

## Build a clean benchmark subset

**Goal:** keep only high-quality tool-use episodes before synthetic hallucination injection.

**Important:** ToolACE has 11,300 dialogues but only ~1,357 extractable tool episodes → **~1,347** clean rows. 

In [8]:
import json
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Any

TOOL_LIST_MARKER = "Here is a list of functions in JSON format that you can invoke:\n"

DATA_DIR = Path("data")
CLEAN_SUBSET_PATH = DATA_DIR / "toolace_clean_subset.jsonl"


@dataclass
class FilterConfig:
    min_query_chars: int = 15
    min_answer_chars: int = 50
    min_tool_output_chars: int = 20
    require_tool_name_in_call: bool = True
    reject_refusals: bool = True


def parse_tools_from_system(system_prompt: str) -> list[dict[str, Any]]:
    """
    Extract available tools from ToolACE system prompt.

    ToolACE stores tool definitions inside the `system` string as JSON.
    We need these tools later for missing-tool hallucination detection.
    """
    if TOOL_LIST_MARKER not in system_prompt:
        return []

    payload = system_prompt.split(TOOL_LIST_MARKER, 1)[1].strip()
    tools, _ = json.JSONDecoder().raw_decode(payload)

    if not isinstance(tools, list):
        raise ValueError(f"Expected list of tools, got {type(tools)}")

    return tools


def is_tool_call(text: str) -> bool:
    """
    Detect whether an assistant message is a tool call.

    Example tool call:
        [Market Trends API(trend_type="MARKET_INDEXES", country="us")]

    Tool outputs often start with JSON like:
        [{"name": "...", "results": ...}]

    So we reject strings starting with '[{' because those are usually tool outputs,
    not assistant tool calls.
    """
    text = text.strip()

    if not text.startswith("["):
        return False

    if "]" not in text:
        return False

    if text.startswith("[{") or text.startswith("[\n{"):
        return False

    return True


def find_previous_tool_call(conversations: list[dict], tool_idx: int) -> int | None:
    """
    For a given tool output turn, find the assistant tool-call turn before it.
    """
    for idx in range(tool_idx - 1, -1, -1):
        turn = conversations[idx]

        if turn["from"] != "assistant":
            continue

        if is_tool_call(turn["value"]):
            return idx

        # If we hit a normal assistant answer before a tool call,
        # this tool output probably does not belong to a clean pattern.
        return None

    return None


def find_previous_user(conversations: list[dict], start_idx: int) -> int | None:
    """
    Find the user query that triggered the tool call.
    """
    for idx in range(start_idx - 1, -1, -1):
        if conversations[idx]["from"] == "user":
            return idx

    return None


def find_next_assistant_answer(conversations: list[dict], tool_idx: int) -> int | None:
    """
    Find the assistant's final natural-language answer after the tool output.
    """
    for idx in range(tool_idx + 1, len(conversations)):
        turn = conversations[idx]

        if turn["from"] != "assistant":
            continue

        if not is_tool_call(turn["value"]):
            return idx

    return None


def extract_tool_episodes(conversations: list[dict]) -> list[dict[str, str]]:
    """
    Extract clean tool-use episodes from one ToolACE dialogue.

    One episode has this structure:
        user -> assistant(tool_call) -> tool(tool_output) -> assistant(answer)
    """
    episodes = []
    #go over all turns
    for tool_idx, turn in enumerate(conversations):
        if turn["from"] != "tool":
            continue

        call_idx = find_previous_tool_call(conversations, tool_idx)
        if call_idx is None:
            continue

        user_idx = find_previous_user(conversations, call_idx)
        if user_idx is None:
            continue

        answer_idx = find_next_assistant_answer(conversations, tool_idx)
        if answer_idx is None:
            continue

        episodes.append(
            {
                "user_query": conversations[user_idx]["value"],
                "tool_call": conversations[call_idx]["value"],
                "tool_output": conversations[tool_idx]["value"],
                "assistant_answer": conversations[answer_idx]["value"],
            }
        )

    return episodes


def parse_tool_output(raw_tool_output: str) -> list | None:
    """
    Tool outputs should be JSON lists like:
        [{"name": "...", "results": {...}}]
    """
    try:
        data = json.loads(raw_tool_output)
    except json.JSONDecodeError:
        return None

    if not isinstance(data, list) or not data:
        return None

    return data


def tool_output_has_results(parsed_output: list) -> bool:
    """
    Keep only tool outputs that contain non-empty results.
    """
    for item in parsed_output:
        if not isinstance(item, dict):
            continue

        results = item.get("results")

        if results not in (None, {}, []):
            return True

    return False


def call_matches_available_tools(tool_call: str, tools: list[dict]) -> bool:
    """
    Check that the called tool name appears in available tools.
    """
    tool_names = [tool.get("name", "") for tool in tools if tool.get("name")]

    if not tool_names:
        return True

    return any(name in tool_call for name in tool_names)


def is_refusal(answer: str) -> bool:
    """
    Reject assistant answers that are mostly refusals.
    """
    answer_head = answer[:250].lower()

    refusal_phrases = (
        "i cannot",
        "i can't",
        "unable to",
        "no function",
        "cannot fulfill",
        "don't have access",
    )

    return any(phrase in answer_head for phrase in refusal_phrases)


def get_episode_quality_issues(
    episode: dict[str, str],
    tools: list[dict],
    cfg: FilterConfig,
) -> list[str]:
    """
    Return rejection reasons.
    Empty list means the episode is clean.
    """
    issues = []

    query = episode["user_query"].strip()
    answer = episode["assistant_answer"].strip()
    tool_output = episode["tool_output"].strip()
    tool_call = episode["tool_call"].strip()

    if len(query) < cfg.min_query_chars:
        issues.append("short_query")

    if len(answer) < cfg.min_answer_chars:
        issues.append("short_answer")

    if len(tool_output) < cfg.min_tool_output_chars:
        issues.append("short_tool_output")

    if not is_tool_call(tool_call):
        issues.append("invalid_tool_call")

    parsed_output = parse_tool_output(tool_output)

    if parsed_output is None:
        issues.append("invalid_tool_json")
    elif not tool_output_has_results(parsed_output):
        issues.append("empty_tool_results")

    if cfg.require_tool_name_in_call:
        if tools and not call_matches_available_tools(tool_call, tools):
            issues.append("call_tool_mismatch")

    if cfg.reject_refusals and is_refusal(answer):
        issues.append("refusal")

    return issues


def episode_to_clean_row(
    episode: dict[str, str],
    tools: list[dict],
    dialogue_idx: int,
    episode_idx: int,
) -> dict[str, Any]:
    """
    Convert extracted ToolACE episode into RAGTruth-style clean row.

    RAGTruth-style mapping:
        query   = user query
        context = tool output
        output  = assistant answer
    """
    tool_names = [tool.get("name", "") for tool in tools if tool.get("name")]

    return {
        "id": f"{dialogue_idx}_ep{episode_idx}",
        "query": episode["user_query"],
        "context": episode["tool_output"],
        "output": episode["assistant_answer"],
        "tool_call": episode["tool_call"],
        "available_tools": json.dumps(tool_names, ensure_ascii=False),
        "available_tools_json": json.dumps(tools, ensure_ascii=False),
        "tool_names": tool_names,
        "hallucination_labels": "[]",
        "dialogue_idx": dialogue_idx,
        "episode_idx": episode_idx,
    }


def make_dedupe_key(episode: dict[str, str]) -> str:
    """
    Build a lightweight deduplication key.

    We do not need exact full-text dedupe here.
    Near-identical query + tool output prefix is enough.
    """
    query_part = episode["user_query"][:200]
    tool_part = episode["tool_output"][:400]

    return f"{query_part}|||{tool_part}"


def build_clean_subset(
    dataset,
    cfg: FilterConfig | None = None,
    target_size: int | None = None,
) -> tuple[list[dict[str, Any]], Counter]:
    """
    Main Stage 1a function.

    It scans ToolACE dialogues, extracts tool-use episodes,
    filters low-quality ones, deduplicates them,
    and returns clean benchmark rows.
    """
    cfg = cfg or FilterConfig()

    clean_rows = []
    rejection_stats = Counter()
    seen_keys = set()

    for dialogue_idx, dialogue in enumerate(dataset):
        tools = parse_tools_from_system(dialogue["system"])
        episodes = extract_tool_episodes(dialogue["conversations"])

        for episode_idx, episode in enumerate(episodes):
            issues = get_episode_quality_issues(episode, tools, cfg)

            if issues:
                rejection_stats.update(issues)
                continue

            dedupe_key = make_dedupe_key(episode)

            if dedupe_key in seen_keys:
                rejection_stats["duplicate"] += 1
                continue

            seen_keys.add(dedupe_key)

            clean_rows.append(
                episode_to_clean_row(
                    episode=episode,
                    tools=tools,
                    dialogue_idx=dialogue_idx,
                    episode_idx=episode_idx,
                )
            )

            if target_size is not None and len(clean_rows) >= target_size:
                rejection_stats["truncated_to_target"] += 1
                return clean_rows, rejection_stats

    return clean_rows, rejection_stats


def save_jsonl(rows: list[dict], path: Path) -> None:
    """
    Save clean rows as JSONL.
    One row = one future hallucination benchmark sample.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def print_subset_report(dataset, clean_rows: list[dict], rejection_stats: Counter) -> None:
    """
    Print simple sanity checks.
    """
    extracted_count = sum(
        len(extract_tool_episodes(row["conversations"]))
        for row in dataset
    )

    print(f"ToolACE dialogues loaded: {len(dataset)}")
    print(f"Extracted tool-use episodes: {extracted_count}")
    print(f"Clean subset size: {len(clean_rows)}")
    print(f"Rejection breakdown: {dict(rejection_stats)}")

    if not clean_rows:
        return

    query_lens = [len(row["query"]) for row in clean_rows]
    context_lens = [len(row["context"]) for row in clean_rows]
    output_lens = [len(row["output"]) for row in clean_rows]

    print()
    print("Example row keys:")
    print(list(clean_rows[0].keys()))

    print()
    print("Example:")
    print("id:", clean_rows[0]["id"])
    print("query:", clean_rows[0]["query"][:160], "...")
    print("context chars:", len(clean_rows[0]["context"]))
    print("output chars:", len(clean_rows[0]["output"]))

    print()
    print(
        "query chars min/median/max:",
        min(query_lens),
        sorted(query_lens)[len(query_lens) // 2],
        max(query_lens),
    )
    print(
        "context chars min/median/max:",
        min(context_lens),
        sorted(context_lens)[len(context_lens) // 2],
        max(context_lens),
    )
    print(
        "output chars min/median/max:",
        min(output_lens),
        sorted(output_lens)[len(output_lens) // 2],
        max(output_lens),
    )

In [9]:
cfg = FilterConfig()
clean_subset, rejection_stats = build_clean_subset(
    dataset=toolace,
    cfg=cfg,
    target_size=None,
)

save_jsonl(clean_subset, CLEAN_SUBSET_PATH)

print_subset_report(
    dataset=toolace,
    clean_rows=clean_subset,
    rejection_stats=rejection_stats,
)

print(f"\nSaved to: {CLEAN_SUBSET_PATH.resolve()}")

ToolACE dialogues loaded: 11300
Extracted tool-use episodes: 1357
Clean subset size: 1347
Rejection breakdown: {'refusal': 2, 'short_answer': 7, 'duplicate': 1}

Example row keys:
['id', 'query', 'context', 'output', 'tool_call', 'available_tools', 'available_tools_json', 'tool_names', 'hallucination_labels', 'dialogue_idx', 'episode_idx']

Example:
id: 0_ep0
query: I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US? ...
context chars: 784
output chars: 793

query chars min/median/max: 22 182 1107
context chars min/median/max: 51 209 5725
output chars min/median/max: 55 283 4016

Saved to: /gpfs/home/slukina/personal/hallucination/data/toolace_clean_subset.jsonl


### clean ToolACE subset

This stage converts raw ToolACE dialogues into RAGTruth-style clean rows for later synthetic hallucination injection.

| Item | Value |
|------|-------|
| ToolACE dialogues loaded | **11,300** |
| Dialogues with at least one `tool` turn | **797** |
| Total `tool` turns | **1,367** |
| Extractable tool-use episodes | **1,357** |
| Tool turns not linked into complete episodes | **10** |
| Clean subset after quality filters | **1,347** |
| Rejections after extraction | **7 short answers, 2 refusals, 1 duplicate** |
| 10k clean target | **Not reachable from ToolACE alone without augmentation** |

Constructed clean row schema: `id`, `query`, `context`, `output`, `tool_call`, `available_tools`, `available_tools_json`, `tool_names`, `hallucination_labels`, `dialogue_idx`, `episode_idx`.

Here, `query`, `context`, and `output` follow the RAGTruth-style meaning: user request, grounding context, and assistant answer. The other fields are tool-calling metadata added for this project.

Saved artifact: `data/toolace_clean_subset.jsonl`, expected line count: **1,347**.

Next stage: corrupt `output` into contradiction, overgeneration, and missing-tool variants, then fill `hallucination_labels` with character-offset spans.

###  Synthetic corruption (3 dataset types)

**Run order:** shared utils (cell below) → contradiction → overgeneration.

| Type | Method | Output file |
|------|--------|-------------|
| Contradiction | LLM replaces one grounded fact with a conflicting fact | `data/toolace_contradiction.jsonl` |
| Overgeneration | LLM adds one unsupported phrase | `data/toolace_overgeneration.jsonl` |
| Missing tool | LLM adds unavailable capability | `data/toolace_missing_tool.jsonl` |


In [10]:
# LLM corruption utilities
# !pip install -q openai python-dotenv tqdm

import json
import os
import re
from collections.abc import Callable
from pathlib import Path
from typing import Any

from openai import OpenAI
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment or in .env")

client = OpenAI()
CLEAN_SUBSET_PATH = Path("data/toolace_clean_subset.jsonl")
LLM_MODEL = "gpt-4.1-mini"
CHECKPOINT_EVERY = 50
RESUME_FROM_CHECKPOINT = True


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def save_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def append_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def parse_llm_json(text: str) -> dict[str, Any]:
    text = text.strip()
    fence = re.match(r"^```(?:json)?\s*(.*?)```\s*$", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()
    return json.loads(text)


def call_llm_json(prompt: str, temperature: float = 0.7) -> dict[str, Any]:
    response = client.responses.create(
        model=LLM_MODEL,
        input=prompt,
        temperature=temperature,
    )
    return parse_llm_json(response.output_text)


def find_unique_span_offsets(text: str, span: str) -> tuple[int, int] | None:
    if text.count(span) != 1:
        return None
    start = text.find(span)
    return start, start + len(span)


def _sentence_spans_for_validation(text: str) -> list[tuple[int, int]]:
    """Non-empty sentence spans (start, end) with whitespace trimmed."""
    spans: list[tuple[int, int]] = []
    pattern = re.compile(r"[^.!?\n]+[.!?]?", flags=re.MULTILINE)
    for match in pattern.finditer(text):
        start, end = match.start(), match.end()
        sent = text[start:end]
        leading = len(sent) - len(sent.lstrip())
        trailing = len(sent.rstrip())
        s2 = start + leading
        e2 = start + trailing
        if e2 > s2:
            spans.append((s2, e2))
    return spans


def span_is_last_sentence(output: str, span: str) -> bool:
    offsets = find_unique_span_offsets(output, span)
    if offsets is None:
        return False
    sents = _sentence_spans_for_validation(output)
    if not sents:
        return False
    last_start, last_end = sents[-1]
    start, end = offsets
    return start == last_start and end == last_end


def span_ends_at_answer_end(output: str, span: str) -> bool:
    offsets = find_unique_span_offsets(output, span)
    if offsets is None:
        return False
    _start, end = offsets
    return end == len(output) or output[end:].strip() == ""


def is_tail_span_placement(output: str, span: str) -> bool:
    """True when the hallucinated span is the last sentence or sits at the answer tail."""
    return span_is_last_sentence(output, span) or span_ends_at_answer_end(output, span)


class TailPlacementQuota:
    """Allow natural end placement, but cap how often the dataset uses tail shortcuts."""

    def __init__(self, max_tail_ratio: float = 0.45, min_samples: int = 25):
        self.max_tail_ratio = max_tail_ratio
        self.min_samples = min_samples
        self.accepted = 0
        self.tail = 0

    def allows(self, output: str, span: str) -> bool:
        if not is_tail_span_placement(output, span):
            return True
        if self.accepted < self.min_samples:
            return True
        return (self.tail + 1) / (self.accepted + 1) <= self.max_tail_ratio

    def register(self, output: str, span: str) -> None:
        self.accepted += 1
        if is_tail_span_placement(output, span):
            self.tail += 1

    def summary(self) -> str:
        if not self.accepted:
            return "n=0"
        return f"tail={self.tail}/{self.accepted} ({self.tail / self.accepted:.1%})"


def build_corrupted_row(
    row: dict[str, Any],
    *,
    corruption_type: str,
    corrupted_output: str,
    hallucinated_span: str,
    label_type: str,
    meta: str,
) -> dict[str, Any] | None:
    offsets = find_unique_span_offsets(corrupted_output, hallucinated_span)
    if offsets is None:
        return None
    start, end = offsets
    label = {
        "start": start,
        "end": end,
        "text": hallucinated_span,
        "label_type": label_type,
        "meta": meta,
    }
    return {
        **row,
        "id": f"{row['id']}_{corruption_type}",
        "source_id": row["id"],
        "clean_output": row["output"],
        "output": corrupted_output,
        "corruption_type": corruption_type,
        "hallucination_labels": json.dumps([label], ensure_ascii=False),
    }


def load_done_source_ids(*paths: Path) -> set[str]:
    done: set[str] = set()
    for path in paths:
        if not path.exists():
            continue
        for row in load_jsonl(path):
            done.add(row.get("source_id", row.get("id", "")))
    return done


def run_llm_corruption_batch(
    *,
    rows: list[dict[str, Any]],
    make_row_fn: Callable[[dict[str, Any]], dict[str, Any] | None],
    final_path: Path,
    checkpoint_path: Path,
    desc: str,
    max_rows: int | None = None,
    checkpoint_every: int = CHECKPOINT_EVERY,
    resume: bool = RESUME_FROM_CHECKPOINT,
) -> tuple[list[dict[str, Any]], int]:
    rows_to_process = rows if max_rows is None else rows[:max_rows]

    if resume and final_path.exists() and not checkpoint_path.exists():
        checkpoint_path.write_bytes(final_path.read_bytes())

    done_source_ids: set[str] = set()
    if resume:
        done_source_ids = load_done_source_ids(checkpoint_path, final_path)

    pending_rows = [r for r in rows_to_process if r["id"] not in done_source_ids]
    failed = 0
    batch_buffer: list[dict[str, Any]] = []
    processed_since_checkpoint = 0

    print(f"Rows to process this run: {len(pending_rows)} (already done: {len(done_source_ids)})")

    for row in tqdm(pending_rows, desc=desc, unit="row"):
        corrupted = make_row_fn(row)
        if corrupted is not None:
            batch_buffer.append(corrupted)
        else:
            failed += 1

        processed_since_checkpoint += 1
        if processed_since_checkpoint >= checkpoint_every:
            append_jsonl(batch_buffer, checkpoint_path)
            batch_buffer.clear()
            processed_since_checkpoint = 0

    append_jsonl(batch_buffer, checkpoint_path)
    batch_buffer.clear()

    all_rows = load_jsonl(checkpoint_path) if checkpoint_path.exists() else []
    save_jsonl(all_rows, final_path)
    return all_rows, failed

In [ ]:
CONTRADICTION_PATH = Path("data/toolace_contradiction.jsonl")
CONTRADICTION_CHECKPOINT_PATH = Path("data/toolace_contradiction.checkpoint.jsonl")
MAX_CONTRADICTION_ROWS: int | None = None  # None = all clean rows


def build_contradiction_prompt(row: dict[str, Any]) -> str:
    return f"""
You are creating a synthetic hallucination benchmark for tool-calling.

Given:
1. USER QUERY
2. TOOL OUTPUT
3. CLEAN ASSISTANT ANSWER

Your task:
Create a CONTRADICTION hallucination by changing exactly ONE short factual span in the assistant answer.

Assignment-style example:
Answer: The weather in Beijing is rainy. [hallucination]
(when the tool output says sunny)

What counts as contradiction:
- The clean answer says a fact supported by the tool output.
- You replace that fact with a different fact that conflicts with the tool output.
- The change can be a single word or short phrase (not necessarily a whole new sentence).
- Examples: sunny -> rainy, available -> unavailable, approved -> rejected, Beijing -> Shanghai, 4172.80 -> 5172.80, +0.68% -> -0.68%.

Strict rules:
- Change exactly ONE factual span.
- Keep the rest of the answer unchanged as much as possible.
- Do NOT add new unsupported sentences.
- Do NOT delete useful content.
- Do NOT rewrite the whole answer.
- The hallucinated span must appear exactly once in corrupted_output.
- Return valid JSON only.

Return this JSON:
{{
  "corrupted_output": "...",
  "original_supported_span": "the exact original text that was changed",
  "hallucinated_span": "the exact new contradictory text in corrupted_output",
  "reason": "short explanation of why it contradicts the tool output"
}}

USER QUERY:
{row["query"]}

TOOL OUTPUT:
{row["context"]}

CLEAN ASSISTANT ANSWER:
{row["output"]}
""".strip()


def make_llm_contradiction_row(row: dict[str, Any]) -> dict[str, Any] | None:
    clean_output = row["output"]

    try:
        llm_result = call_llm_json(build_contradiction_prompt(row), temperature=0.4)
    except Exception as e:
        print(f"LLM failed for {row['id']}: {e}")
        return None

    corrupted_output = llm_result["corrupted_output"]
    original_span = llm_result["original_supported_span"]
    hallucinated_span = llm_result["hallucinated_span"]
    reason = llm_result.get("reason", "")

    if original_span not in clean_output:
        print(f"Original span not found in clean output for {row['id']}")
        return None

    return build_corrupted_row(
        row,
        corruption_type="contradiction",
        corrupted_output=corrupted_output,
        hallucinated_span=hallucinated_span,
        label_type="Evident Conflict",
        meta=f"Contradiction injected. Original supported span: {original_span!r}. {reason}",
    )


clean_rows = load_jsonl(CLEAN_SUBSET_PATH)
contradiction_rows, contradiction_failed = run_llm_corruption_batch(
    rows=clean_rows,
    make_row_fn=make_llm_contradiction_row,
    final_path=CONTRADICTION_PATH,
    checkpoint_path=CONTRADICTION_CHECKPOINT_PATH,
    desc="Contradiction",
    max_rows=MAX_CONTRADICTION_ROWS,
)

print(f"Clean rows loaded: {len(clean_rows)}")
print(f"Contradiction rows total: {len(contradiction_rows)}")
print(f"Failed / skipped this run: {contradiction_failed}")
print(f"Checkpoint: {CONTRADICTION_CHECKPOINT_PATH.resolve()}")
print(f"Final file: {CONTRADICTION_PATH.resolve()}")


Rows to process this run: 61 (already done: 1286)


Contradiction:   0%|          | 0/61 [00:00<?, ?row/s]

Original span not found in clean output for 176_ep0
Original span not found in clean output for 180_ep1
Original span not found in clean output for 253_ep0
Original span not found in clean output for 384_ep0
Original span not found in clean output for 388_ep0
Original span not found in clean output for 425_ep0
Original span not found in clean output for 721_ep0
Clean rows loaded: 1347
Contradiction rows total: 1291
Failed / skipped this run: 56
Checkpoint: /gpfs/home/slukina/personal/hallucination/data/toolace_contradiction.checkpoint.jsonl
Final file: /gpfs/home/slukina/personal/hallucination/data/toolace_contradiction.jsonl


In [11]:
import json
from pathlib import Path

CONTRADICTION_PATH = Path("data/toolace_contradiction.jsonl")
SUSPICIOUS_PATH = Path("data/toolace_contradiction.suspicious.jsonl")

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

rows = load_jsonl(CONTRADICTION_PATH)

suspicious = []

for row in rows:
    labels = json.loads(row["hallucination_labels"])
    label = labels[0]
    span = label["text"]

    if span in row["clean_output"] or span in row["context"]:
        suspicious.append(row)

save_jsonl(suspicious, SUSPICIOUS_PATH)

print("suspicious contradiction rows:", len(suspicious))
print("saved to:", SUSPICIOUS_PATH.resolve())

for row in suspicious[:10]:
    label = json.loads(row["hallucination_labels"])[0]
    print("=" * 100)
    print("ID:", row["id"])
    print("SPAN:", repr(label["text"]))
    print("META:", label.get("meta", ""))
    print("CLEAN:", row["clean_output"][:500].replace("\n", " "))
    print("CORR:", row["output"][:500].replace("\n", " "))
    print("CONTEXT:", row["context"][:500].replace("\n", " "))

suspicious contradiction rows: 46
saved to: /gpfs/home/slukina/personal/hallucination/data/toolace_contradiction.suspicious.jsonl
ID: 42_ep0_contradiction
SPAN: 'ZMW'
META: Contradiction injected. Original supported span: 'ZWL'. The tool output lists 'ZWL' as an available currency symbol, but the corrupted answer incorrectly ends with 'ZMW' instead, contradicting the tool output.
CLEAN: The available currency symbols have been successfully retrieved. Here is the comprehensive list of currency symbols you can use for your analysis:  - AED - AFN - ALL - AMD - ANG - AOA - ARS - AUD - AWG - AZN - BAM - BBD - BDT - BGN - BHD - BIF - BMD - BND - BOB - BRL - BSD - BTN - BWP - BYN - BZD - CAD - CDF - CHF - CLP - CNY - COP - CRC - CUP - CVE - CZK - DJF - DKK - DOP - DZD - EGP - ERN - ETB - EUR - FJD - FKP - FOK - GBP - GEL - GGP - GHS - GIP - GMD - GNF - GTQ - GYD - HKD - HNL - HRK - HT
CORR: The available currency symbols have been successfully retrieved. Here is the comprehensive list of curr

In [12]:
# Overgeneration corruption — uses shared utils from the previous cell.
OVERGEN_PATH = Path("data/toolace_overgeneration.jsonl")
OVERGEN_CHECKPOINT_PATH = Path("data/toolace_overgeneration.checkpoint.jsonl")
MAX_OVERGEN_ROWS: int | None = None  # None = all clean rows

# Set True once after changing prompt/filters, then re-run to regenerate JSONL.
RESET_OVERGEN_OUTPUTS = False

# Allow natural end placement, but avoid "always last sentence" shortcuts (~55% tail max).
OVERGEN_TAIL_QUOTA = TailPlacementQuota(max_tail_ratio=0.55, min_samples=25)

In [10]:

def build_overgeneration_prompt(row: dict[str, Any]) -> str:
    return f"""
You are creating a synthetic hallucination benchmark.

Given:
1. USER QUERY
2. TOOL OUTPUT
3. CLEAN ASSISTANT ANSWER

Your task:
Add exactly ONE short unsupported phrase to the assistant answer.

Preferred style (assignment example — comma-appended clause):
Answer: The weather in Beijing is sunny, and the weather has been pretty good the past few months [overgeneration]

You may also insert a separate short sentence elsewhere when that reads more naturally.

Rules:
- Keep the original answer almost unchanged.
- Do not remove existing correct information.
- Add only one unsupported claim.
- The added claim must NOT be supported by the tool output.
- Prefer a natural comma-appended extra clause when the answer already states the main tool-supported fact.
- If you use a comma, include the comma inside hallucinated_span exactly as in corrupted_output.
- Vary placement: after the first sentence, mid-answer, or end — not always the last sentence.
- hallucinated_span must be exactly the added unsupported text as it appears in corrupted_output.
- hallucinated_span must appear exactly once in corrupted_output.
- Return valid JSON only.
- JSON keys must be:
  - corrupted_output
  - hallucinated_span

USER QUERY:
{row["query"]}

TOOL OUTPUT:
{row["context"]}

CLEAN ASSISTANT ANSWER:
{row["output"]}
""".strip()


def make_overgeneration_row(row: dict[str, Any]) -> dict[str, Any] | None:
    try:
        llm_result = call_llm_json(build_overgeneration_prompt(row), temperature=0.7)
    except Exception as e:
        print(f"LLM failed for {row['id']}: {e}")
        return None

    corrupted_output = str(llm_result.get("corrupted_output", "")).strip()
    hallucinated_span = str(llm_result.get("hallucinated_span", "")).strip()
    if not corrupted_output or not hallucinated_span:
        return None
    if hallucinated_span in row["output"]:
        return None
    if not OVERGEN_TAIL_QUOTA.allows(corrupted_output, hallucinated_span):
        return None

    built = build_corrupted_row(
        row,
        corruption_type="overgeneration",
        corrupted_output=corrupted_output,
        hallucinated_span=hallucinated_span,
        label_type="Evident Baseless Info",
        meta="Overgeneration: this statement is plausible but unsupported by the tool output.",
    )
    if built is not None:
        OVERGEN_TAIL_QUOTA.register(corrupted_output, hallucinated_span)
    return built


if RESET_OVERGEN_OUTPUTS:
    for path in [OVERGEN_CHECKPOINT_PATH, OVERGEN_PATH]:
        if path.exists():
            path.unlink()
            print(f"Deleted old file: {path}")

clean_rows = load_jsonl(CLEAN_SUBSET_PATH)
overgeneration_rows, overgen_failed = run_llm_corruption_batch(
    rows=clean_rows,
    make_row_fn=make_overgeneration_row,
    final_path=OVERGEN_PATH,
    checkpoint_path=OVERGEN_CHECKPOINT_PATH,
    desc="Overgeneration",
    max_rows=MAX_OVERGEN_ROWS,
)

print(f"Clean rows loaded: {len(clean_rows)}")
print(f"Overgeneration rows total: {len(overgeneration_rows)}")
print(f"Failed / skipped this run: {overgen_failed}")
print(f"Checkpoint: {OVERGEN_CHECKPOINT_PATH.resolve()}")
print(f"Final file: {OVERGEN_PATH.resolve()}")
print(f"Tail placement: {OVERGEN_TAIL_QUOTA.summary()}")


Rows to process this run: 2 (already done: 1345)


Overgeneration:   0%|          | 0/2 [00:00<?, ?row/s]

Clean rows loaded: 1347
Overgeneration rows total: 1347
Failed / skipped this run: 0
Checkpoint: /gpfs/home/slukina/personal/hallucination/data/toolace_overgeneration.checkpoint.jsonl
Final file: /gpfs/home/slukina/personal/hallucination/data/toolace_overgeneration.jsonl
Tail placement: tail=0/2 (0.0%)


In [12]:
# Stage 1b (3/3): Missing-tool — LLM adds one phrase/clause claiming an unavailable external capability.
# Run the shared "LLM corruption utilities" cell first.
# It must define: load_jsonl, call_llm_json, build_corrupted_row, run_llm_corruption_batch.

import json
from pathlib import Path
from typing import Any


MISSING_TOOL_PATH = Path("data/toolace_missing_tool.jsonl")
MISSING_TOOL_CHECKPOINT_PATH = Path("data/toolace_missing_tool.checkpoint.jsonl")

# Use 50 for debugging. Set to None for all clean rows (~1347).
MAX_MISSING_TOOL_ROWS: int | None = None

# Set True once after changing prompt/filters, then re-run to regenerate JSONL.
RESET_MISSING_TOOL_OUTPUTS = False

# Allow natural end placement, but cap tail shortcuts (~40% tail max for missing_tool).
MISSING_TOOL_TAIL_QUOTA = TailPlacementQuota(max_tail_ratio=0.40, min_samples=25)

# Retry bad generations before giving up on a row.
MAX_MISSING_TOOL_RETRIES = 3

MAX_MISSING_TOOL_SPAN_CHARS = 130


STRONG_MISSING_TOOL_PATTERNS = [
    "send",
    "email",
    "book",
    "ticket",
    "hotel",
    "appointment",
    "reservation",
    "schedule",
    "calendar",
    "reminder",
    "place",
    "trade",
    "order",
    "buy",
    "purchase",
    "sell",
    "transfer",
    "funds",
    "money",
    "save",
    "account",
    "alarm",
    "register",
    "upload",
    "download",
    "call",
    "message",
    "cancel",
    "subscription",
]

WEAK_LLM_ACTION_PATTERNS = [
    "explain",
    "summarize",
    "analyze",
    "compare",
    "recommend",
    "translate",
    "describe",
    "calculate",
    "advise",
    "answer",
    "help you understand",
]

AGGRESSIVE_ACTION_PATTERNS = [
    "top-performing stocks",
    "promotional emails",
    "potential customers",
    "immediately invest",
    "invest in",
    "buy stocks",
    "sell stocks",
    "transfer the funds immediately",
    "transfer money immediately",
]

MISSING_TOOL_OPENINGS = [
    "i can",
    "i can also",
    "would you like me to",
    "shall i",
    "do you want me to",
    "i'll",
    "i will",
]

# Factual-style claims (harder to catch with offer-regex heuristics).
MISSING_TOOL_CLAIM_PATTERNS = [
    " was sent",
    " has been sent",
    " have been sent",
    " was scheduled",
    " has been scheduled",
    " was booked",
    " has been booked",
    " was registered",
    " has been registered",
    " was uploaded",
    " has been uploaded",
    " was saved",
    " has been saved",
    " confirmation ",
    " reminder ",
    " your account",
    " to your email",
]


def _tool_names_for_prompt(row: dict[str, Any]) -> list[str]:
    """Normalize tool_names from clean subset."""
    tool_names = row.get("tool_names")

    if isinstance(tool_names, str):
        try:
            tool_names = json.loads(tool_names)
        except Exception:
            tool_names = [tool_names]

    if not tool_names:
        tool_names = json.loads(row["available_tools"])

    return [str(t) for t in tool_names]


def has_allowed_missing_tool_opening(span: str) -> bool:
    """Allow offers/clauses with or without a leading comma."""
    s = span.lower().strip().lstrip(",").strip()
    if any(s.startswith(opening) for opening in MISSING_TOOL_OPENINGS):
        return True
    return any(
        phrase in span.lower()
        for phrase in (
            "would you like me to",
            "would you like to",
            "do you want me to",
            "shall i ",
        )
    )


def is_good_missing_tool_span(span: str) -> bool:
    """
    Keep only concise, realistic missing-tool offers/actions.

    Missing-tool means the assistant offers or claims an external action
    requiring a tool that is not available.
    """
    s = span.lower().strip()
    words = s.split()

    if len(span) > MAX_MISSING_TOOL_SPAN_CHARS:
        return False

    if len(words) < 4 or len(words) > 20:
        return False

    has_offer = has_allowed_missing_tool_opening(span)
    has_claim = any(p in s for p in MISSING_TOOL_CLAIM_PATTERNS)
    if not (has_offer or has_claim):
        return False

    if any(p in s for p in WEAK_LLM_ACTION_PATTERNS):
        return False

    if any(p in s for p in AGGRESSIVE_ACTION_PATTERNS):
        return False

    if not any(p in s for p in STRONG_MISSING_TOOL_PATTERNS):
        return False

    return True


def _missing_tool_placement_hint(row_id: str) -> str:
    hints = [
        "Prefer comma-appending a short clause to an existing sentence (e.g. '..., would you like me to book a ticket?').",
        "Insert a short separate sentence after the first paragraph when that sounds natural.",
        "Insert a short separate sentence in the middle when that sounds natural.",
        "Comma-append or add a short closing offer only if that is the most natural option.",
    ]
    return hints[hash(row_id) % len(hints)]


def build_missing_tool_prompt(row: dict[str, Any]) -> str:
    tool_names = _tool_names_for_prompt(row)
    placement = _missing_tool_placement_hint(str(row.get("id", "")))

    return f"""
You are creating a synthetic hallucination benchmark for tool-calling.

Given:
1. USER QUERY
2. AVAILABLE TOOLS
3. TOOL OUTPUT
4. CLEAN ASSISTANT ANSWER

Your task:
Insert exactly ONE new sentence into the assistant answer where the assistant claims or implies an external real-world action that requires an unavailable tool.

This is a missing-tool hallucination.

Placement hint: {placement}

A valid missing-tool hallucination MUST claim or offer an external action or side effect, such as:
- sending an email or message
- booking a ticket, hotel, appointment, or reservation
- scheduling a calendar event or reminder
- placing a trade or order
- buying or selling something
- transferring money or funds
- saving something to a user account
- setting an alarm on a device
- registering the user for an event
- uploading or downloading a file
- calling someone
- canceling a reservation or subscription

Do NOT use normal language-model abilities, such as:
- explain
- summarize
- analyze
- compare
- recommend
- translate
- describe
- calculate
- answer questions
- provide advice

Strict rules:
- Add exactly ONE new phrase or clause (not multiple claims).
- hallucinated_span must be 4 to 18 words and under 130 characters.
- Short comma-appended offers are valid (e.g. ", would you like me to book a ticket?").
- The phrase must describe an external action requiring an unavailable tool.
- Use varied wording. About half the time, write a factual claim (not a question), e.g.:
  - "A confirmation email has been sent to your inbox."
  - "Your reminder was scheduled for tomorrow morning."
- About half the time, a natural offer is also acceptable, e.g.:
  - "Would you like me to book a ticket?"
  - "I can send this report to your email."
- Do NOT rely only on templates like "Would you like me to" or "I can".
- Vary placement: beginning, middle, or end are all allowed when natural.
- Do not put every example at the very end; mix positions across examples.
- Prefer neutral administrative actions:
  send, email, book, schedule, register, save, upload, download, call, cancel.
- Avoid aggressive financial, marketing, or irreversible actions.
- Do NOT say things like:
  - investing in top-performing stocks
  - sending promotional emails
  - transferring funds immediately
- Do NOT mention the missing tool explicitly.
- Do NOT add more than one new claim.
- Keep the original answer unchanged except for the one inserted phrase/clause.
- hallucinated_span must be exactly the added text as it appears in corrupted_output (include a leading comma if comma-appending).
- hallucinated_span must appear exactly once in corrupted_output.
- Return valid JSON only.

Return this JSON:
{{
  "corrupted_output": "...",
  "hallucinated_span": "...",
  "missing_tool_reason": "which unavailable external action is claimed"
}}

USER QUERY:
{row["query"]}

AVAILABLE TOOLS:
{json.dumps(tool_names, ensure_ascii=False)}

TOOL OUTPUT:
{row["context"]}

CLEAN ASSISTANT ANSWER:
{row["output"]}
""".strip()


def make_missing_tool_row(row: dict[str, Any]) -> dict[str, Any] | None:
    clean_output = row["output"]

    for attempt in range(1, MAX_MISSING_TOOL_RETRIES + 1):
        try:
            llm_result = call_llm_json(
                build_missing_tool_prompt(row),
                temperature=0.4,
            )
        except Exception as e:
            print(f"LLM failed for {row['id']} attempt {attempt}: {e}")
            continue

        corrupted_output = llm_result.get("corrupted_output", "").strip()
        hallucinated_span = llm_result.get("hallucinated_span", "").strip()
        reason = llm_result.get("missing_tool_reason", "").strip()

        if not corrupted_output or not hallucinated_span:
            continue

        if corrupted_output == clean_output:
            continue

        if hallucinated_span in clean_output:
            continue

        if not is_good_missing_tool_span(hallucinated_span):
            continue

        if corrupted_output.count(hallucinated_span) != 1:
            continue

        if not MISSING_TOOL_TAIL_QUOTA.allows(corrupted_output, hallucinated_span):
            continue

        built = build_corrupted_row(
            row,
            corruption_type="missing_tool",
            corrupted_output=corrupted_output,
            hallucinated_span=hallucinated_span,
            label_type="Missing Tool",
            meta=f"Missing-tool hallucination: claims/offers an unavailable external capability. {reason}",
        )
        if built is not None:
            MISSING_TOOL_TAIL_QUOTA.register(corrupted_output, hallucinated_span)
            return built
        continue

    print(f"Failed after retries for {row['id']}")
    return None


if RESET_MISSING_TOOL_OUTPUTS:
    for path in [MISSING_TOOL_CHECKPOINT_PATH, MISSING_TOOL_PATH]:
        if path.exists():
            path.unlink()
            print(f"Deleted old file: {path}")


clean_rows = load_jsonl(CLEAN_SUBSET_PATH)

missing_tool_rows, missing_failed = run_llm_corruption_batch(
    rows=clean_rows,
    make_row_fn=make_missing_tool_row,
    final_path=MISSING_TOOL_PATH,
    checkpoint_path=MISSING_TOOL_CHECKPOINT_PATH,
    desc="Missing-tool",
    max_rows=MAX_MISSING_TOOL_ROWS,
)

print(f"Clean rows loaded: {len(clean_rows)}")
print(f"Missing-tool rows total: {len(missing_tool_rows)}")
print(f"Failed / skipped this run: {missing_failed}")
print(f"Checkpoint: {MISSING_TOOL_CHECKPOINT_PATH.resolve()}")
print(f"Final file: {MISSING_TOOL_PATH.resolve()}")
print(f"Tail placement: {MISSING_TOOL_TAIL_QUOTA.summary()}")

Rows to process this run: 506 (already done: 841)


Missing-tool:   0%|          | 0/506 [00:00<?, ?row/s]

Failed after retries for 4_ep1
Failed after retries for 14_ep1
Failed after retries for 21_ep0
Failed after retries for 21_ep1
Failed after retries for 35_ep0
Failed after retries for 46_ep0
Failed after retries for 47_ep0
Failed after retries for 53_ep0
Failed after retries for 61_ep1
Failed after retries for 91_ep1
Failed after retries for 103_ep0
Failed after retries for 103_ep1
Failed after retries for 105_ep0
Failed after retries for 109_ep0
Failed after retries for 112_ep0
Failed after retries for 120_ep1
Failed after retries for 143_ep1
Failed after retries for 176_ep0
Failed after retries for 196_ep1
Failed after retries for 206_ep1
Failed after retries for 211_ep1
Failed after retries for 225_ep1
Failed after retries for 229_ep1
Failed after retries for 286_ep0
Failed after retries for 287_ep1
Failed after retries for 327_ep1
Failed after retries for 331_ep1
Failed after retries for 364_ep0
Failed after retries for 368_ep1
Failed after retries for 369_ep0
Failed after retries 

In [13]:
# Quality audit for corrupted datasets: overgeneration + contradiction + missing-tool.
from collections import Counter
from difflib import SequenceMatcher

CONTRADICTION_PATH = Path("data/toolace_contradiction.jsonl")
OVERGEN_PATH = Path("data/toolace_overgeneration.jsonl")
MISSING_TOOL_PATH = Path("data/toolace_missing_tool.jsonl")


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def audit_corruption_rows(rows: list, name: str, corruption_type: str) -> None:
    """Print structural QA stats for one corrupted dataset."""
    if not rows:
        print(f"\n=== {name}: no rows (check path or run corruption cell first) ===")
        return

    sims = [similarity(r["clean_output"], r["output"]) for r in rows]
    ratios = []
    span_in_clean = 0
    span_in_context = 0
    broken_offsets = 0
    identical_output = 0
    counter = Counter()

    for row in rows:
        if row["clean_output"] == row["output"]:
            identical_output += 1

        labels = json.loads(row["hallucination_labels"])

        for label in labels:
            text = label["text"]
            start = label["start"]
            end = label["end"]

            counter[text] += 1
            ratios.append((end - start) / max(len(row["output"]), 1))

            if row["output"][start:end] != text:
                broken_offsets += 1
            if text in row["clean_output"]:
                span_in_clean += 1
            if text in row["context"]:
                span_in_context += 1

    print(f"\n{'=' * 60}")
    print(f"{name}  (n={len(rows)}, type={corruption_type})")
    print(f"{'=' * 60}")
    print(
        f"min / median / mean similarity(clean, corrupted): "
        f"{min(sims):.3f} / {sorted(sims)[len(sims)//2]:.3f} / {sum(sims)/len(sims):.3f}"
    )
    print(f"median hallucination span ratio: {sorted(ratios)[len(ratios)//2]:.3f}")
    print(f"identical clean_output == output: {identical_output}")
    print(f"broken span offsets: {broken_offsets}")
    print(f"spans already in clean_output: {span_in_clean}")
    print(f"spans substring of context: {span_in_context}")

    if corruption_type == "overgeneration":
        print("(expect: high similarity, span NOT in clean_output/context)")
    elif corruption_type == "contradiction":
        print("(expect: high similarity; span is the NEW conflicting value, may appear elsewhere in context)")
    elif corruption_type == "missing_tool":
        print("(expect: high similarity; span is a new external action claim, NOT in clean_output/context)")

    print("top repeated span texts:")
    for text, count in counter.most_common(10):
        preview = text.replace("\n", " ")[:80]
        print(f"  {count}x  {preview!r}")

In [14]:
contradiction_rows = load_jsonl(CONTRADICTION_PATH) if CONTRADICTION_PATH.exists() else []
overgeneration_rows = load_jsonl(OVERGEN_PATH) if OVERGEN_PATH.exists() else []
missing_tool_rows = load_jsonl(MISSING_TOOL_PATH) if MISSING_TOOL_PATH.exists() else []

print("Loaded:")
print("contradiction:", len(contradiction_rows))
print("overgeneration:", len(overgeneration_rows))
print("missing_tool:", len(missing_tool_rows))

audit_corruption_rows(overgeneration_rows, "Overgeneration", "overgeneration")
audit_corruption_rows(contradiction_rows, "Contradiction", "contradiction")
audit_corruption_rows(missing_tool_rows, "Missing-tool", "missing_tool")

Loaded:
contradiction: 1291
overgeneration: 1347
missing_tool: 1178

Overgeneration  (n=1347, type=overgeneration)
min / median / mean similarity(clean, corrupted): 0.028 / 0.925 / 0.903
median hallucination span ratio: 0.131
identical clean_output == output: 0
broken span offsets: 0
spans already in clean_output: 0
spans substring of context: 0
(expect: high similarity, span NOT in clean_output/context)
top repeated span texts:
  4x  'which is updated daily'
  3x  'which is expected to continue growing rapidly'
  3x  'and it is expected to rise steadily over the next quarter'
  2x  'and it is available in 4K resolution'
  2x  'which are updated weekly'
  2x  'which is unusually high'
  2x  'which are updated daily'
  2x  'which is expected to rise significantly in the next quarter'
  2x  'which is expected to rise soon'
  2x  'which is known for its vibrant colors'

Contradiction  (n=1291, type=contradiction)
min / median / mean similarity(clean, corrupted): 0.170 / 0.991 / 0.967
medi

In [15]:
#  span annotations and save four RAGTruth-style datasets:
#   3 corrupted datasets + 1 clean episode-level dataset.
#   Query  = user query
#   Context = tool response
#   Output = model final answer
#   hallucination_labels = span-labeled hallucinations added to the initial dataset

import json
from pathlib import Path
from typing import Any
from collections import Counter

CONTRADICTION_PATH = Path("data/toolace_contradiction.jsonl")
OVERGEN_PATH = Path("data/toolace_overgeneration.jsonl")
MISSING_TOOL_PATH = Path("data/toolace_missing_tool.jsonl")
CLEAN_SUBSET_PATH = Path("data/toolace_clean_subset.jsonl")

OUT_DIR = Path("data/ragtruth_style")

RAGTRUTH_CONTRADICTION_PATH = OUT_DIR / "toolace_contradiction_ragtruth.jsonl"
RAGTRUTH_OVERGEN_PATH = OUT_DIR / "toolace_overgeneration_ragtruth.jsonl"
RAGTRUTH_MISSING_TOOL_PATH = OUT_DIR / "toolace_missing_tool_ragtruth.jsonl"
RAGTRUTH_CLEAN_PATH = OUT_DIR / "toolace_clean_ragtruth.jsonl"


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def save_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def parse_labels(row: dict[str, Any]) -> list[dict[str, Any]]:
    labels = row.get("hallucination_labels", "[]")

    if isinstance(labels, str):
        labels = json.loads(labels)

    if not isinstance(labels, list):
        raise ValueError(
            f"hallucination_labels must be list or JSON-string list, got {type(labels)}"
        )

    return labels


def validate_span_annotations(row: dict[str, Any]) -> list[str]:
    """
    Validate RAGTruth-style character spans.

    Important:
    start/end are character offsets inside row["output"],
    not inside input_str, query, or context.
    """
    issues = []
    output = row.get("output", "")
    labels = parse_labels(row)

    if not labels:
        issues.append("empty_labels")

    for i, label in enumerate(labels):
        start = label.get("start")
        end = label.get("end")
        text = label.get("text")

        if not isinstance(start, int) or not isinstance(end, int):
            issues.append(f"label_{i}_non_integer_offsets")
            continue

        if start < 0 or end > len(output) or start >= end:
            issues.append(f"label_{i}_bad_offset_range")
            continue

        if output[start:end] != text:
            issues.append(f"label_{i}_broken_span_text")

    return issues


def build_input_str(row: dict[str, Any]) -> str:
    """
    Optional packed input string for baselines.
    This is metadata, not required by the assignment.
    """
    return f"""
USER QUERY:
{row["query"]}

CONTEXT / TOOL RESPONSE:
{row["context"]}

OUTPUT / ASSISTANT FINAL ANSWER:
{row["output"]}
""".strip()


def to_ragtruth_style(row: dict[str, Any], *, default_corruption_type: str | None = None) -> dict[str, Any]:
    """
    Convert one ToolACE row to the assignment-required RAGTruth-style format.

    Required by assignment:
        query
        context
        output
        hallucination_labels

    Extra metadata is kept only because it is useful for debugging and later experiments.
    """
    labels = parse_labels(row)
    source_id = row.get("source_id") or row["id"]
    clean_output = row.get("clean_output") or row["output"]

    return {
        "id": row["id"],
        "query": row["query"],
        "context": row["context"],
        "output": row["output"],
        "hallucination_labels": json.dumps(labels, ensure_ascii=False),

        # Optional but useful metadata
        "corruption_type": row.get("corruption_type", default_corruption_type),
        "source_id": source_id,
        "clean_output": clean_output,
        "tool_call": row.get("tool_call"),
        "available_tools": row.get("available_tools"),
        "tool_names": row.get("tool_names"),
        "dialogue_idx": row.get("dialogue_idx"),
        "episode_idx": row.get("episode_idx"),
        "input_str": row.get("input_str") or build_input_str(row),
    }


def validate_and_convert_dataset(
    input_path: Path,
    output_path: Path,
    expected_type: str,
) -> list[dict[str, Any]]:
    raw_rows = load_jsonl(input_path)

    issues_counter = Counter()
    valid_rows = []

    for row in raw_rows:
        if row.get("corruption_type") != expected_type:
            issues_counter["wrong_corruption_type"] += 1
            continue

        issues = validate_span_annotations(row)

        if issues:
            for issue in issues:
                issues_counter[issue] += 1
            continue

        valid_rows.append(to_ragtruth_style(row))

    save_jsonl(valid_rows, output_path)

    print("=" * 80)
    print(f"{expected_type}")
    print(f"input path:  {input_path}")
    print(f"output path: {output_path}")
    print(f"raw rows:    {len(raw_rows)}")
    print(f"valid rows:  {len(valid_rows)}")
    print(f"issues:      {dict(issues_counter)}")

    return valid_rows


def validate_and_convert_clean_dataset(
    input_path: Path,
    output_path: Path,
) -> list[dict[str, Any]]:
    raw_rows = load_jsonl(input_path)

    issues_counter = Counter()
    valid_rows = []

    for row in raw_rows:
        labels = parse_labels(row)
        if labels:
            issues_counter["clean_has_labels"] += 1
            continue

        clean_row = dict(row)
        clean_row["source_id"] = row.get("source_id") or row["id"]
        clean_row["clean_output"] = row.get("clean_output") or row["output"]
        clean_row["corruption_type"] = "clean"

        valid_rows.append(to_ragtruth_style(clean_row, default_corruption_type="clean"))

    save_jsonl(valid_rows, output_path)

    print("=" * 80)
    print("clean")
    print(f"input path:  {input_path}")
    print(f"output path: {output_path}")
    print(f"raw rows:    {len(raw_rows)}")
    print(f"valid rows:  {len(valid_rows)}")
    print(f"issues:      {dict(issues_counter)}")

    return valid_rows


contradiction_rows = validate_and_convert_dataset(
    input_path=CONTRADICTION_PATH,
    output_path=RAGTRUTH_CONTRADICTION_PATH,
    expected_type="contradiction",
)

overgeneration_rows = validate_and_convert_dataset(
    input_path=OVERGEN_PATH,
    output_path=RAGTRUTH_OVERGEN_PATH,
    expected_type="overgeneration",
)

missing_tool_rows = validate_and_convert_dataset(
    input_path=MISSING_TOOL_PATH,
    output_path=RAGTRUTH_MISSING_TOOL_PATH,
    expected_type="missing_tool",
)

clean_rows = validate_and_convert_clean_dataset(
    input_path=CLEAN_SUBSET_PATH,
    output_path=RAGTRUTH_CLEAN_PATH,
)

print("=" * 80)
print("Final Stage 3–4 artifacts:")
print(f"Contradiction:  {len(contradiction_rows)} rows -> {RAGTRUTH_CONTRADICTION_PATH.resolve()}")
print(f"Overgeneration: {len(overgeneration_rows)} rows -> {RAGTRUTH_OVERGEN_PATH.resolve()}")
print(f"Missing-tool:   {len(missing_tool_rows)} rows -> {RAGTRUTH_MISSING_TOOL_PATH.resolve()}")
print(f"Clean:          {len(clean_rows)} rows -> {RAGTRUTH_CLEAN_PATH.resolve()}")

contradiction
input path:  data/toolace_contradiction.jsonl
output path: data/ragtruth_style/toolace_contradiction_ragtruth.jsonl
raw rows:    1291
valid rows:  1291
issues:      {}
overgeneration
input path:  data/toolace_overgeneration.jsonl
output path: data/ragtruth_style/toolace_overgeneration_ragtruth.jsonl
raw rows:    1347
valid rows:  1347
issues:      {}
missing_tool
input path:  data/toolace_missing_tool.jsonl
output path: data/ragtruth_style/toolace_missing_tool_ragtruth.jsonl
raw rows:    1178
valid rows:  1178
issues:      {}
clean
input path:  data/toolace_clean_subset.jsonl
output path: data/ragtruth_style/toolace_clean_ragtruth.jsonl
raw rows:    1347
valid rows:  1347
issues:      {}
Final Stage 3–4 artifacts:
Contradiction:  1291 rows -> /gpfs/home/slukina/personal/hallucination/data/ragtruth_style/toolace_contradiction_ragtruth.jsonl
Overgeneration: 1347 rows -> /gpfs/home/slukina/personal/hallucination/data/ragtruth_style/toolace_overgeneration_ragtruth.jsonl
Missi

### Mixed evaluation set

- **Hallucinated rows:** 3816 (three corruption JSONL files; canonical count after dedup).
- **Clean rows:** 1347 unique episodes (`toolace_clean_ragtruth.jsonl`).
- **Mixed total:** 5163 rows.
- **Split:** by `source_id`, **70% / 10% / 20%** (train / validation / test).
- **Downstream:** `mixed_train`, `mixed_val`, `mixed_test`; per-type test views in `datasets['contradiction'|...]`.

See **Section 2 — Data & Methodology** for full tables and diagnostics.


In [16]:
import json
import random
from pathlib import Path
from typing import Any

import pandas as pd

DATA_DIR = Path("data/ragtruth_style")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

HALLUCINATION_PATHS = {
  "contradiction": DATA_DIR / "toolace_contradiction_ragtruth.jsonl",
  "overgeneration": DATA_DIR / "toolace_overgeneration_ragtruth.jsonl",
  "missing_tool": DATA_DIR / "toolace_missing_tool_ragtruth.jsonl",
}
CLEAN_DATASET_PATH = DATA_DIR / "toolace_clean_ragtruth.jsonl"


def load_jsonl(path: Path) -> list[dict[str, Any]]:
  """Load one JSONL dataset."""
  with path.open("r", encoding="utf-8") as f:
    return [json.loads(line) for line in f]


def write_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
  path.parent.mkdir(parents=True, exist_ok=True)
  with path.open("w", encoding="utf-8") as f:
    for row in rows:
      f.write(json.dumps(row, ensure_ascii=False) + "\n")


def parse_maybe_json(value: Any) -> Any:
  if isinstance(value, str):
    try:
      return json.loads(value)
    except json.JSONDecodeError:
      return value
  return value


def normalize_type(raw_type: Any) -> str | None:
  if raw_type is None:
    return None
  s = str(raw_type).strip().lower()
  if s in {"contradiction", "tool_output_conflict", "tool_output_contradiction", "conflict"}:
    return "contradiction"
  if s in {"overgeneration", "unsupported_extra", "unsupported_information"}:
    return "overgeneration"
  if s in {"missing_tool", "missing_tool_action", "missing_tool_action_recommendation"}:
    return "missing_tool"
  return s


def parse_gold_spans(row: dict[str, Any]) -> list[dict[str, Any]]:
  """Gold spans; character offsets inside row['output']. Empty on clean rows."""
  labels = row.get("hallucination_labels", [])
  if isinstance(labels, str):
    labels = json.loads(labels)
  return labels


def normalize_labels_for_row(row: dict[str, Any], hallucination_type: str) -> list[dict[str, Any]]:
  output = str(row["output"])
  raw_labels = parse_maybe_json(row.get("hallucination_labels", []))
  if isinstance(raw_labels, dict):
    raw_labels = [raw_labels]
  if not raw_labels:
    return []

  fixed = []
  for lab in raw_labels:
    s, e = int(lab["start"]), int(lab["end"])
    if not (0 <= s <= e <= len(output)):
      raise ValueError(f"Bad span offsets for {row.get('id')}: {s}, {e}, len={len(output)}")
    text = output[s:e]
    fixed.append({"start": s, "end": e, "text": text, "type": hallucination_type})
  return fixed


def load_hallucinated_raw_datasets() -> list[dict[str, Any]]:
  rows = []
  for htype, path in HALLUCINATION_PATHS.items():
    if not path.exists():
      raise FileNotFoundError(f"Missing dataset file: {path}")
    for i, row in enumerate(load_jsonl(path), start=1):
      row = dict(row)
      row["_dataset_name"] = htype
      row["_source_file"] = path.name
      row["_line_idx"] = i
      rows.append(row)
  return rows


def build_hallucinated_records(raw_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
  records = []
  for row in raw_rows:
    htype = normalize_type(row.get("corruption_type")) or normalize_type(row.get("_dataset_name"))
    labels = normalize_labels_for_row(row, htype)
    records.append({
      "id": row["id"],
      "source_id": str(row["source_id"]),
      "split": None,
      "query": row["query"],
      "context": row["context"],
      "output": row["output"],
      "clean_output": row["clean_output"],
      "hallucination_labels": labels,
      "hallucination_type": htype,
      "corruption_type": row.get("corruption_type"),
      "tool_call": row.get("tool_call"),
      "available_tools": parse_maybe_json(row.get("available_tools")),
      "tool_names": parse_maybe_json(row.get("tool_names")),
      "dialogue_idx": row.get("dialogue_idx"),
      "episode_idx": row.get("episode_idx"),
      "input_str": row.get("input_str"),
      "source_file": row["_source_file"],
      "line_idx": int(row["_line_idx"]),
      "variant": "hallucinated",
      "is_hallucinated": 1,
    })
  return records


def build_clean_records(raw_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
  records = []
  for i, row in enumerate(raw_rows, start=1):
    labels = parse_gold_spans(row)
    if labels:
      raise ValueError(f"Clean row {row.get('id')} unexpectedly has labels")

    source_id = str(row.get("source_id") or row["id"])
    output = row["output"]

    records.append({
      "id": row["id"],
      "source_id": source_id,
      "split": None,
      "query": row["query"],
      "context": row["context"],
      "output": output,
      "clean_output": row.get("clean_output") or output,
      "hallucination_labels": [],
      "hallucination_type": "clean",
      "corruption_type": "clean",
      "tool_call": row.get("tool_call"),
      "available_tools": parse_maybe_json(row.get("available_tools")),
      "tool_names": parse_maybe_json(row.get("tool_names")),
      "dialogue_idx": row.get("dialogue_idx"),
      "episode_idx": row.get("episode_idx"),
      "input_str": row.get("input_str"),
      "source_file": CLEAN_DATASET_PATH.name,
      "line_idx": i,
      "variant": "clean",
      "is_hallucinated": 0,
    })
  return records


def build_split_map(source_ids: set[str], seed: int = RANDOM_SEED) -> dict[str, str]:
  ordered = sorted(source_ids)
  random.Random(seed).shuffle(ordered)

  n_train = int(0.70 * len(ordered))
  n_val = int(0.10 * len(ordered))
  train_sids = set(ordered[:n_train])
  val_sids = set(ordered[n_train : n_train + n_val])
  test_sids = set(ordered[n_train + n_val :])

  split_map = {}
  for sid in ordered:
    if sid in train_sids:
      split_map[sid] = "train"
    elif sid in val_sids:
      split_map[sid] = "validation"
    else:
      split_map[sid] = "test"
  return split_map


def assign_splits(records: list[dict[str, Any]], split_map: dict[str, str]) -> list[dict[str, Any]]:
  for r in records:
    sid = str(r["source_id"])
    if sid not in split_map:
      raise ValueError(f"Missing split for source_id={sid}")
    r["split"] = split_map[sid]
  return records


# --- load original eval corpora ---
datasets_hallucinated: dict[str, list[dict[str, Any]]] = {}
for name, path in HALLUCINATION_PATHS.items():
  rows = load_jsonl(path)
  datasets_hallucinated[name] = rows
  print(f"{name}: {len(rows)} hallucinated rows from {path}")

clean_raw = load_jsonl(CLEAN_DATASET_PATH)
print(f"clean: {len(clean_raw)} unique clean rows from {CLEAN_DATASET_PATH}")

# --- build hallucinated + clean episode-level mixed splits ---
raw_rows = load_hallucinated_raw_datasets()
canonical_all = build_hallucinated_records(raw_rows)
clean_all = build_clean_records(clean_raw)

split_map = build_split_map({str(r["source_id"]) for r in clean_all}, seed=RANDOM_SEED)
canonical_all = assign_splits(canonical_all, split_map)
clean_all = assign_splits(clean_all, split_map)

mixed_all = canonical_all + clean_all

hallucinated_splits = {
  split: [r for r in canonical_all if r["split"] == split]
  for split in ["train", "validation", "test"]
}
clean_splits = {
  split: [r for r in clean_all if r["split"] == split]
  for split in ["train", "validation", "test"]
}
mixed_splits = {
  split: [r for r in mixed_all if r["split"] == split]
  for split in ["train", "validation", "test"]
}

# Per-type hallucinated test views stay useful for qualitative examples.
datasets = {
  htype: [r for r in hallucinated_splits["test"] if r["hallucination_type"] == htype]
  for htype in sorted({r["hallucination_type"] for r in canonical_all})
}

mixed_train = mixed_splits["train"]
mixed_val = mixed_splits["validation"]
mixed_test = mixed_splits["test"]
clean_test = clean_splits["test"]

write_jsonl(canonical_all, PROCESSED_DIR / "toolace_generative_all_fixed.jsonl")
write_jsonl(clean_all, PROCESSED_DIR / "toolace_clean_all_fixed.jsonl")
write_jsonl(mixed_all, PROCESSED_DIR / "toolace_generative_mixed_all.jsonl")
for split, rows in mixed_splits.items():
  write_jsonl(rows, PROCESSED_DIR / f"toolace_generative_mixed_{split}.jsonl")

print("\nCanonical hallucinated rows:", len(canonical_all))
print("Unique clean rows:", len(clean_all))
print("Mixed rows (hallucinated + unique clean):", len(mixed_all))
print("Mixed split x is_hallucinated:")
print(pd.crosstab([r["split"] for r in mixed_all], [r["is_hallucinated"] for r in mixed_all]))
print("\nSaved under:", PROCESSED_DIR.resolve())
print("Downstream defaults:")
print("- `datasets` = hallucinated TEST per type")
print("- `mixed_test` = logical mixed TEST (unique clean + hallucinated)")
print("- `mixed_train` / `mixed_val` = logical mixed train/validation")
for htype, rows in datasets.items():
  print(f"  {htype}: {len(rows)} hallucinated test rows")
print(f"  clean_test: {len(clean_test)} unique clean test rows")
print(f"  mixed_test: {len(mixed_test)} total test rows")

contradiction: 1291 hallucinated rows from data/ragtruth_style/toolace_contradiction_ragtruth.jsonl
overgeneration: 1347 hallucinated rows from data/ragtruth_style/toolace_overgeneration_ragtruth.jsonl
missing_tool: 1178 hallucinated rows from data/ragtruth_style/toolace_missing_tool_ragtruth.jsonl
clean: 1347 unique clean rows from data/ragtruth_style/toolace_clean_ragtruth.jsonl

Canonical hallucinated rows: 3816
Unique clean rows: 1347
Mixed rows (hallucinated + unique clean): 5163
Mixed split x is_hallucinated:
col_0         0     1
row_0                
test        271   761
train       942  2675
validation  134   380

Saved under: /gpfs/home/slukina/personal/hallucination/data/ragtruth_style/processed
Downstream defaults:
- `datasets` = hallucinated TEST per type
- `mixed_test` = logical mixed TEST (unique clean + hallucinated)
- `mixed_train` / `mixed_val` = logical mixed train/validation
  contradiction: 257 hallucinated test rows
  missing_tool: 233 hallucinated test rows
  ov

### Dataset summary (code section)

Counts and splits are documented in **Section 2 — Data & Methodology** (tables above). Re-run the mixed-dataset cell to refresh numbers if the JSONL files change.

Current defaults: `mixed_test` = **1032** rows (257 + 233 + 271 corrupted test + 271 clean).


In [17]:

from lettuce_baseline import char_level_prf, parse_gold_spans, trim_spans_whitespace

def merge_overlapping_spans(spans: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Merge overlapping or touching spans after offset normalization."""
    clean: list[tuple[int, int]] = []

    for span in spans or []:
        start = int(span["start"])
        end = int(span["end"])
        if end > start:
            clean.append((start, end))

    if not clean:
        return []

    clean.sort()
    merged = [[clean[0][0], clean[0][1]]]
    for start, end in clean[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    return [{"start": start, "end": end} for start, end in merged]


def normalize_pred_spans(pred_spans: Any, output_text: str) -> list[dict[str, Any]]:
    """Sanitize detector spans before evaluation."""
    n = len(output_text)
    normalized = []

    for span in pred_spans or []:
        if isinstance(span, dict):
            start, end = span.get("start"), span.get("end")
        else:
            try:
                start, end = span
            except (TypeError, ValueError):
                continue

        try:
            start = max(0, min(int(start), n))
            end = max(0, min(int(end), n))
        except (TypeError, ValueError):
            continue

        if end > start:
            normalized.append({"start": start, "end": end})

    return merge_overlapping_spans(normalized)


def evaluate_detector(
    rows: list[dict[str, Any]],
    detector,
    max_rows: int | None = None,
) -> dict[str, float]:
    """Evaluate detector.predict(row) with micro character-level P/R/F1."""
    subset = rows if max_rows is None else rows[:max_rows]
    totals = {"tp": 0, "fp": 0, "fn": 0}

    for row in subset:
        output = row["output"]
        gold_spans = trim_spans_whitespace(output, parse_gold_spans(row))
        pred_spans = normalize_pred_spans(detector.predict(row), output)
        pred_spans = trim_spans_whitespace(output, pred_spans)

        metrics = char_level_prf(
            pred_spans=pred_spans,
            gold_spans=gold_spans,
            text_len=len(output),
        )
        totals["tp"] += metrics["tp"]
        totals["fp"] += metrics["fp"]
        totals["fn"] += metrics["fn"]

    precision = (
        totals["tp"] / (totals["tp"] + totals["fp"])
        if (totals["tp"] + totals["fp"])
        else 0.0
    )
    recall = (
        totals["tp"] / (totals["tp"] + totals["fn"])
        if (totals["tp"] + totals["fn"])
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n": len(subset),
    }

In [18]:
# Dataset artifact diagnostics (adapted from Updated_Hallucination_Detection_Pipeline).
#
# Purpose: check whether corruption patterns make evaluation too easy.
# - missing_tool is designed to append one sentence at the END -> last_sentence heuristic
#   may look artificially strong on that type.
# - overgeneration does NOT require end placement -> last_sentence is a useful sanity check.
# - diff_vs_clean uses clean_output at inference time (CHEATING upper bound only).

import re
from difflib import SequenceMatcher


def sentence_spans(text: str) -> list[dict[str, Any]]:
    spans = []
    pattern = re.compile(r"[^.!?\n]+[.!?]?", flags=re.MULTILINE)
    for match in pattern.finditer(text):
        start, end = match.start(), match.end()
        sent = text[start:end]
        leading = len(sent) - len(sent.lstrip())
        trailing = len(sent.rstrip())
        s2 = start + leading
        e2 = start + trailing
        if e2 > s2:
            spans.append({"start": s2, "end": e2, "text": text[s2:e2]})
    return spans


def last_sentence_baseline(row: dict[str, Any]) -> list[dict[str, Any]]:
    sents = sentence_spans(row["output"])
    if not sents:
        return []
    last = sents[-1]
    if len(last["text"].strip()) < 20:
        return []
    return [{"start": last["start"], "end": last["end"]}]


ACTION_KEYWORDS = [
    "schedule", "reminder", "remind", "book", "reserve", "reservation",
    "send", "email", "mail", "notify", "notification", "alert",
    "set up", "create", "add", "calendar", "save", "export", "download",
    "share", "purchase", "buy", "order", "subscribe", "call", "contact",
    "message", "text", "sms", "register", "sign up", "appointment",
]
OFFER_PREFIXES = [
    "would you like me to", "do you want me to", "should i", "shall i",
    "can i", "i can also", "i can", "i could", "would you also like me to",
    "would you like help", "let me know if you want me to",
]


def is_missing_tool_offer_sentence(sent: str) -> bool:
    s = " ".join(sent.lower().strip().split())
    harmless = [
        "is there anything else you would like to know",
        "let me know if you need any more information",
        "feel free to ask",
        "hope this helps",
    ]
    if any(h in s for h in harmless):
        return False
    return any(p in s for p in OFFER_PREFIXES) and any(k in s for k in ACTION_KEYWORDS)


def missing_tool_regex_baseline(row: dict[str, Any]) -> list[dict[str, Any]]:
    preds = []
    for sent in sentence_spans(row["output"]):
        if is_missing_tool_offer_sentence(sent["text"]):
            preds.append({"start": sent["start"], "end": sent["end"]})
    return normalize_pred_spans(preds, output_text=row["output"])


def _spans_to_char_set(spans: list[dict[str, Any]]) -> set[int]:
    chars: set[int] = set()
    for span in spans or []:
        chars.update(range(int(span["start"]), int(span["end"])))
    return chars


def diff_vs_clean_baseline(row: dict[str, Any], min_chars: int = 1) -> list[dict[str, Any]]:
    """CHEATING: uses clean_output, not available to real detectors."""
    clean = str(row.get("clean_output", ""))
    output = str(row.get("output", ""))
    spans = []
    matcher = SequenceMatcher(a=clean, b=output, autojunk=False)
    for tag, _i1, _i2, j1, j2 in matcher.get_opcodes():
        if tag in {"insert", "replace"} and (j2 - j1) >= min_chars:
            spans.append({"start": j1, "end": j2})
    return normalize_pred_spans(spans, output_text=output)


class CallableDetector:
    def __init__(self, fn):
        self.fn = fn

    def predict(self, row: dict[str, Any]) -> list[dict[str, Any]]:
        return self.fn(row)


def gold_span_position_stats(rows: list[dict[str, Any]]) -> dict[str, float]:
    """How often gold labels sit at the end / match the last-sentence heuristic."""
    n = len(rows)
    if n == 0:
        return {"n": 0}

    ends_at_output_end = 0
    exact_last_sentence = 0
    overlaps_last_sentence = 0

    for row in rows:
        output = row["output"]
        gold_spans = trim_spans_whitespace(output, parse_gold_spans(row))
        if not gold_spans:
            continue

        gold = gold_spans[0]
        if gold["end"] == len(output) or output[gold["end"] :].strip() == "":
            ends_at_output_end += 1

        last_pred = trim_spans_whitespace(output, last_sentence_baseline(row))
        if last_pred and gold["start"] == last_pred[0]["start"] and gold["end"] == last_pred[0]["end"]:
            exact_last_sentence += 1

        gold_chars = _spans_to_char_set(gold_spans)
        last_chars = _spans_to_char_set(last_pred)
        if gold_chars and gold_chars <= last_chars:
            overlaps_last_sentence += 1

    return {
        "n": n,
        "pct_gold_ends_at_answer_end": ends_at_output_end / n,
        "pct_gold_exact_last_sentence": exact_last_sentence / n,
        "pct_gold_subset_of_last_sentence": overlaps_last_sentence / n,
    }


def run_artifact_diagnostics(
    dataset_by_type: dict[str, list[dict[str, Any]]],
    combined_test_rows: list[dict[str, Any]] | None = None,
) -> None:
    baselines = {
        "last_sentence_all_no_clean": last_sentence_baseline,
        "missing_tool_offer_regex_no_clean": missing_tool_regex_baseline,
        "diagnostic_diff_vs_clean_min1_CHEATING": lambda r: diff_vs_clean_baseline(r, min_chars=1),
    }

    print("=" * 100)
    print("DATASET ARTIFACT DIAGNOSTICS")
    print("Checks whether simple heuristics explain our synthetic labels.")
    print()

    for corruption_type, rows in dataset_by_type.items():
        stats = gold_span_position_stats(rows)
        print(f"[{corruption_type}] gold span position stats")
        for key, value in stats.items():
            if key == "n":
                print(f"  {key}: {value}")
            else:
                print(f"  {key}: {value:.3f}")

    print("\nHeuristic baseline scores (micro char P/R/F1 via evaluate_detector):")
    eval_sets = dict(dataset_by_type)
    if combined_test_rows is not None:
        eval_sets["mixed_test"] = combined_test_rows

    for split_name, rows in eval_sets.items():
        print(f"\n--- split: {split_name} (n={len(rows)}) ---")
        for baseline_name, fn in baselines.items():
            metrics = evaluate_detector(rows, CallableDetector(fn))
            print(
                f"  {baseline_name}: "
                f"P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={metrics['f1']:.3f}"
            )


# Run after the mixed-dataset cell defines `datasets` and `mixed_test`.
if "datasets" in globals():
    run_artifact_diagnostics(datasets, combined_test_rows=mixed_test)
else:
    print("Run the mixed dataset cell first to populate `datasets` and `mixed_test`.")

DATASET ARTIFACT DIAGNOSTICS
Checks whether simple heuristics explain our synthetic labels.

[contradiction] gold span position stats
  n: 257
  pct_gold_ends_at_answer_end: 0.012
  pct_gold_exact_last_sentence: 0.000
  pct_gold_subset_of_last_sentence: 0.187
[missing_tool] gold span position stats
  n: 233
  pct_gold_ends_at_answer_end: 0.283
  pct_gold_exact_last_sentence: 0.266
  pct_gold_subset_of_last_sentence: 0.408
[overgeneration] gold span position stats
  n: 271
  pct_gold_ends_at_answer_end: 0.026
  pct_gold_exact_last_sentence: 0.004
  pct_gold_subset_of_last_sentence: 0.343

Heuristic baseline scores (micro char P/R/F1 via evaluate_detector):

--- split: contradiction (n=257) ---
  last_sentence_all_no_clean: P=0.029 R=0.160 F1=0.049
  missing_tool_offer_regex_no_clean: P=0.000 R=0.000 F1=0.000


  diagnostic_diff_vs_clean_min1_CHEATING: P=0.907 R=0.216 F1=0.349

--- split: missing_tool (n=233) ---
  last_sentence_all_no_clean: P=0.302 R=0.405 F1=0.346
  missing_tool_offer_regex_no_clean: P=0.361 R=0.031 F1=0.058
  diagnostic_diff_vs_clean_min1_CHEATING: P=0.991 R=0.993 F1=0.992

--- split: overgeneration (n=271) ---
  last_sentence_all_no_clean: P=0.160 R=0.352 F1=0.219
  missing_tool_offer_regex_no_clean: P=0.000 R=0.000 F1=0.000
  diagnostic_diff_vs_clean_min1_CHEATING: P=0.951 R=0.995 F1=0.973

--- split: mixed_test (n=1032) ---
  last_sentence_all_no_clean: P=0.119 R=0.352 F1=0.178
  missing_tool_offer_regex_no_clean: P=0.361 R=0.015 F1=0.028
  diagnostic_diff_vs_clean_min1_CHEATING: P=0.970 R=0.896 F1=0.932


In [19]:
# %pip install -q lettucedetect


## Step 2 — Baseline evaluation (logical mixed clean + hallucinated)

**LettuceDetect** (`lettuce_baseline.py`): one inference pass over `mixed_test`, then micro char P/R/F1 on the full mix. Per-type breakdown: corrupted rows of that type + all test clean (no separate clean-only row).

**LookBackLens** (`lookback_baseline.py`): extract attention features, train token classifier on `mixed_train` with epoch metrics on `mixed_val`, then one predict pass on `mixed_test` (same reporting format as Lettuce).

Mapping: `query` = question, `context` = tool output, `output` = answer. Detectors do **not** use `clean_output`.


In [20]:
# ----- Baseline 1: LettuceDetect -----
# %pip install -q lettucedetect

import importlib
from pathlib import Path

import lettuce_baseline
importlib.reload(lettuce_baseline)

from lettuce_baseline import (
    CHECKPOINTS_DIR,
    LettuceDetectWrapper,
    evaluate_corruption_type_mix,
    evaluate_predictions,
    load_predictions,
    predict_all,
    save_predictions,
    summarize_mixed_test,
)

assert "mixed_test" in globals(), "Run the mixed-dataset cell first."

LETTUCE_PREDS_PATH = CHECKPOINTS_DIR / "lettuce_mixed_test_predictions.jsonl"
# Set True to re-run LLM inference even if predictions file exists.
FORCE_LETTUCE_PREDICT = False

if LETTUCE_PREDS_PATH.exists() and not FORCE_LETTUCE_PREDICT:
    lettuce_predictions = load_predictions(LETTUCE_PREDS_PATH)
    print(f"Loaded LettuceDetect predictions: {LETTUCE_PREDS_PATH.resolve()} ({len(lettuce_predictions)} rows)")
else:
    lettuce_detector = LettuceDetectWrapper(threshold=0.5, output_mode="spans")
    lettuce_predictions = predict_all(mixed_test, lettuce_detector)
    save_predictions(
        lettuce_predictions,
        LETTUCE_PREDS_PATH,
        meta={"detector": "lettucedetect", "split": "mixed_test", "n_rows": len(mixed_test)},
    )
    print(f"Saved LettuceDetect predictions: {LETTUCE_PREDS_PATH.resolve()} ({len(lettuce_predictions)} rows)")

mix = summarize_mixed_test(mixed_test)
print("\n=== LettuceDetect on mixed_test (one pass, micro char P/R/F1) ===")
print(
    f"rows: {mix['n_total']} = corrupted {mix['n_corrupted']} + clean {mix['n_clean']}"
)
print(f"  corrupted by type: {mix['by_type']}")

lettuce_mixed_result = evaluate_predictions(mixed_test, lettuce_predictions)
print(
    f"metrics: P={lettuce_mixed_result['precision']:.4f}  "
    f"R={lettuce_mixed_result['recall']:.4f}  "
    f"F1={lettuce_mixed_result['f1']:.4f}"
)

print(
    "\nPer corruption type (corrupted rows of that type + all test clean; "
    "same predictions, no re-inference):"
)
for row in evaluate_corruption_type_mix(mixed_test, lettuce_predictions):
    print(
        f"  {row['hallucination_type']:22s}  "
        f"n={row['n']:3d} (corrupted={row['n_corrupted']:3d} + clean={row['n_clean']:3d})  "
        f"P={row['precision']:.4f}  R={row['recall']:.4f}  F1={row['f1']:.4f}"
    )


Loaded LettuceDetect predictions: /gpfs/home/slukina/personal/hallucination/data/checkpoints/lettuce_mixed_test_predictions.jsonl (515 rows)

=== LettuceDetect on mixed_test (one pass, micro char P/R/F1) ===
rows: 1032 = corrupted 761 + clean 271
  corrupted by type: {'contradiction': 257, 'missing_tool': 233, 'overgeneration': 271}
metrics: P=0.2390  R=0.3728  F1=0.2913

Per corruption type (corrupted rows of that type + all test clean; same predictions, no re-inference):
  contradiction           n=528 (corrupted=257 + clean=271)  P=0.0524  R=0.2709  F1=0.0879
  missing_tool            n=504 (corrupted=233 + clean=271)  P=0.2354  R=0.3762  F1=0.2896
  overgeneration          n=542 (corrupted=271 + clean=271)  P=0.2028  R=0.4004  F1=0.2693


In [23]:
import random

from lettuce_baseline import char_level_prf, parse_gold_spans, trim_spans_whitespace


def show_prediction_examples(
    rows,
    *,
    detector=None,
    predictions=None,
    n=3,
    seed=42,
):
    """Show gold vs predicted spans. Pass `predictions` dict to avoid re-inference."""
    if detector is None and predictions is None:
        raise ValueError("Provide detector or predictions")

    random.seed(seed)
    sample_rows = random.sample(rows, min(n, len(rows)))

    for i, row in enumerate(sample_rows):
        print("=" * 100)
        print(f"Example {i + 1}")
        print("ID:", row["id"])
        print("TYPE:", row.get("hallucination_type") or row.get("corruption_type"))
        print()

        print("QUERY:")
        print(row["query"][:400])
        print()

        print("CONTEXT:")
        print(row["context"][:500])
        print()

        print("CLEAN OUTPUT (reference only, not used by detector):")
        print(row.get("clean_output", "")[:1000])
        print()

        print("ANSWER (output):")
        print(row["output"][:1000])
        print()

        output = row["output"]
        gold_spans = trim_spans_whitespace(output, parse_gold_spans(row))
        if predictions is not None:
            pred_spans = trim_spans_whitespace(
                output, predictions.get(str(row["id"]), [])
            )
        else:
            pred_spans = trim_spans_whitespace(output, detector.predict(row))

        print("GOLD SPANS:")
        if not gold_spans:
            print("  none")
        else:
            for s in gold_spans:
                print(f"  [{s['start']}:{s['end']}] {repr(s['text'])}")

        print()
        print("PREDICTED SPANS:")
        if not pred_spans:
            print("  none")
        else:
            for s in pred_spans:
                print(f"  [{s['start']}:{s['end']}] {repr(s['text'])}")

        print()
        metrics = char_level_prf(
            pred_spans=pred_spans,
            gold_spans=gold_spans,
            text_len=len(output),
        )
        print("METRICS (this row):")
        print(
            f"  precision={metrics['precision']:.3f}  "
            f"recall={metrics['recall']:.3f}  "
            f"f1={metrics['f1']:.3f}"
        )

In [31]:
show_prediction_examples(
    rows=[r for r in datasets["contradiction"] if str(r["id"]) in lettuce_predictions],
    predictions=lettuce_predictions,
    n=5,
)

show_prediction_examples(
    rows=[r for r in datasets["missing_tool"] if str(r["id"]) in lettuce_predictions],
    predictions=lettuce_predictions,
    n=3,
)

show_prediction_examples(
    rows=[r for r in datasets["overgeneration"] if str(r["id"]) in lettuce_predictions],
    predictions=lettuce_predictions,
    n=3,
)


Example 1
ID: 151_ep0_contradiction
TYPE: contradiction

QUERY:
Hey, can you show me the latest news about the Golden Gate Bridge anniversary event in San Francisco?

CONTEXT:
[{"name": "Get News by Keyword", "results": {"news_articles": [{"title": "Celebrating 85 Years: Golden Gate Bridge Anniversary", "source": "San Francisco Chronicle", "url": "https://www.sfchronicle.com/golden-gate-bridge-85th-anniversary"}, {"title": "Golden Gate Bridge 85th Anniversary: Events and History", "source": "SFGATE", "url": "https://www.sfgate.com/golden-gate-bridge-85th-anniversary-events"}, {"title": "Marking the 85th Anniversary of the Golden Gate Bridge", "source": "abc7News", "ur

CLEAN OUTPUT (reference only, not used by detector):
Great, here are the latest news articles about the Golden Gate Bridge anniversary event in San Francisco:

1. **Celebrating 85 Years: Golden Gate Bridge Anniversary**
   - Source: San Francisco Chronicle
   - [Read more](https://www.sfchronicle.com/golden-gate-bridge-8

In [23]:
# ----- Baseline 2: LookBackLens -----
# pip install transformers torch scikit-learn joblib accelerate

import importlib
import lookback_baseline
importlib.reload(lookback_baseline)

from lookback_baseline import (
    CHECKPOINTS_DIR,
    DEFAULT_MODEL,
    PAPER_MODEL,
    LookBackLensWrapper,
    evaluate_corruption_type_mix,
    evaluate_predictions,
    load_predictions,
    predict_all as lookback_predict_all,
    save_predictions,
    summarize_mixed_test,
)

assert "mixed_train" in globals() and "mixed_val" in globals() and "mixed_test" in globals(), (
    "Run the mixed-dataset cell first."
)

# Paper backbone (needs HF access + ~24GB VRAM for 7B):
# LOOKBACK_MODEL = PAPER_MODEL
LOOKBACK_MODEL = DEFAULT_MODEL
LOOKBACK_EPOCHS = 12

LOOKBACK_CLASSIFIER_PATH = CHECKPOINTS_DIR / "lookback_mixed_classifier.joblib"
LOOKBACK_PREDS_PATH = CHECKPOINTS_DIR / "lookback_mixed_test_predictions.jsonl"
LOOKBACK_TRAIN_FEATURES_PATH = CHECKPOINTS_DIR / "lookback_mixed_train_features.npz"
LOOKBACK_VAL_FEATURES_PATH = CHECKPOINTS_DIR / "lookback_mixed_val_features.npz"
# Set True to re-train and re-run test inference even if checkpoint files exist.
FORCE_LOOKBACK_TRAIN = False
FORCE_LOOKBACK_PREDICT = False

if LOOKBACK_CLASSIFIER_PATH.exists() and not FORCE_LOOKBACK_TRAIN:
    lookback = LookBackLensWrapper.load_artifacts(LOOKBACK_CLASSIFIER_PATH)
    lookback_detector = lookback
    print(f"Loaded LookBack classifier: {LOOKBACK_CLASSIFIER_PATH.resolve()}")
else:
    lookback = LookBackLensWrapper(
        model_name=LOOKBACK_MODEL,
        threshold=0.5,
        sliding_window=8,
        max_length=2048,
        prompt_style="toolace",
    )
    lookback_detector = lookback

    train_mix = summarize_mixed_test(mixed_train)
    val_mix = summarize_mixed_test(mixed_val)
    print("=== LookBackLens training (token-level head; features from attention) ===")
    print(
        f"train rows: {train_mix['n_total']} = corrupted {train_mix['n_corrupted']} + clean {train_mix['n_clean']}"
    )
    print(f"  corrupted by type: {train_mix['by_type']}")
    print(
        f"val rows:   {val_mix['n_total']} = corrupted {val_mix['n_corrupted']} + clean {val_mix['n_clean']}"
    )
    print(f"  corrupted by type: {val_mix['by_type']}")
    print(f"model: {LOOKBACK_MODEL}  epochs: {LOOKBACK_EPOCHS}\n")
    print(f"train feature cache: {LOOKBACK_TRAIN_FEATURES_PATH.resolve()}")
    print(f"val feature cache:   {LOOKBACK_VAL_FEATURES_PATH.resolve()}\n")

    lookback.fit_with_validation(
        mixed_train,
        mixed_val,
        epochs=LOOKBACK_EPOCHS,
        max_train_rows=None,
        max_val_rows=None,
        train_cache_path=LOOKBACK_TRAIN_FEATURES_PATH,
        val_cache_path=LOOKBACK_VAL_FEATURES_PATH,
    )
    lookback.save_artifacts(LOOKBACK_CLASSIFIER_PATH)
    print(f"Saved LookBack classifier: {LOOKBACK_CLASSIFIER_PATH.resolve()}")

if LOOKBACK_PREDS_PATH.exists() and not FORCE_LOOKBACK_PREDICT:
    lookback_predictions = load_predictions(LOOKBACK_PREDS_PATH)
    print(f"Loaded LookBack predictions: {LOOKBACK_PREDS_PATH.resolve()} ({len(lookback_predictions)} rows)")
else:
    if "lookback" not in globals():
        lookback = LookBackLensWrapper.load_artifacts(LOOKBACK_CLASSIFIER_PATH)
        lookback_detector = lookback
    lookback_predictions = lookback_predict_all(mixed_test, lookback)
    save_predictions(
        lookback_predictions,
        LOOKBACK_PREDS_PATH,
        meta={
            "detector": "lookback_lens",
            "model_name": LOOKBACK_MODEL,
            "split": "mixed_test",
            "n_rows": len(mixed_test),
            "epochs": LOOKBACK_EPOCHS,
        },
    )
    print(f"Saved LookBack predictions: {LOOKBACK_PREDS_PATH.resolve()} ({len(lookback_predictions)} rows)")

test_mix = summarize_mixed_test(mixed_test)
print("\n=== LookBackLens on mixed_test (one predict pass, micro char P/R/F1) ===")
print(
    f"rows: {test_mix['n_total']} = corrupted {test_mix['n_corrupted']} + clean {test_mix['n_clean']}"
)
print(f"  corrupted by type: {test_mix['by_type']}")

lookback_mixed_result = evaluate_predictions(mixed_test, lookback_predictions)
print(
    f"metrics: P={lookback_mixed_result['precision']:.4f}  "
    f"R={lookback_mixed_result['recall']:.4f}  "
    f"F1={lookback_mixed_result['f1']:.4f}"
)

print(
    "\nPer corruption type (corrupted rows of that type + all test clean; "
    "same predictions, no re-inference):"
)
for row in evaluate_corruption_type_mix(mixed_test, lookback_predictions):
    print(
        f"  {row['hallucination_type']:22s}  "
        f"n={row['n']:3d} (corrupted={row['n_corrupted']:3d} + clean={row['n_clean']:3d})  "
        f"P={row['precision']:.4f}  R={row['recall']:.4f}  F1={row['f1']:.4f}"
    )


=== LookBackLens training (token-level head; features from attention) ===
train rows: 3617 = corrupted 2675 + clean 942
  corrupted by type: {'contradiction': 905, 'missing_tool': 828, 'overgeneration': 942}
val rows:   514 = corrupted 380 + clean 134
  corrupted by type: {'contradiction': 129, 'missing_tool': 117, 'overgeneration': 134}
model: TinyLlama/TinyLlama-1.1B-Chat-v1.0  epochs: 12

train feature cache: /gpfs/home/slukina/personal/hallucination/data/checkpoints/lookback_mixed_train_features.npz
val feature cache:   /gpfs/home/slukina/personal/hallucination/data/checkpoints/lookback_mixed_val_features.npz



LookBackLens train features:   0%|          | 0/3617 [00:00<?, ?row/s]

Saved LookBackLens train feature cache: /gpfs/home/slukina/personal/hallucination/data/checkpoints/lookback_mixed_train_features.npz


LookBackLens val features:   0%|          | 0/514 [00:00<?, ?row/s]

Saved LookBackLens val feature cache: /gpfs/home/slukina/personal/hallucination/data/checkpoints/lookback_mixed_val_features.npz
Token samples: train=469177 (factual=443979, hallucinated=25198)  val=70300 (factual=66619, hallucinated=3681)
   epoch   train_acc   train_bal     val_acc     val_bal
       1      0.6696      0.7981      0.6718      0.7996
       2      0.7696      0.8413      0.7745      0.8390
       3      0.7859      0.8498      0.7890      0.8453
       4      0.7898      0.8529      0.7903      0.8462
       5      0.7970      0.8561      0.7972      0.8482
       6      0.8069      0.8599      0.8075      0.8520
       7      0.8173      0.8639      0.8187      0.8553
       8      0.8268      0.8669      0.8289      0.8578
       9      0.8364      0.8697      0.8382      0.8600
      10      0.8433      0.8712      0.8454      0.8615
      11      0.8502      0.8734      0.8533      0.8643
      12      0.8558      0.8746      0.8593      0.8654
Saved LookBack clas

LookBackLens predict:   0%|          | 0/1032 [00:00<?, ?row/s]

Saved LookBack predictions: /gpfs/home/slukina/personal/hallucination/data/checkpoints/lookback_mixed_test_predictions.jsonl (1032 rows)

=== LookBackLens on mixed_test (one predict pass, micro char P/R/F1) ===
rows: 1032 = corrupted 761 + clean 271
  corrupted by type: {'contradiction': 257, 'missing_tool': 233, 'overgeneration': 271}
metrics: P=0.2939  R=0.9006  F1=0.4432

Per corruption type (corrupted rows of that type + all test clean; same predictions, no re-inference):
  contradiction           n=528 (corrupted=257 + clean=271)  P=0.0512  R=0.4734  F1=0.0923
  missing_tool            n=504 (corrupted=233 + clean=271)  P=0.3024  R=0.9757  F1=0.4617
  overgeneration          n=542 (corrupted=271 + clean=271)  P=0.2496  R=0.9464  F1=0.3950


In [27]:
show_prediction_examples(
    rows=[r for r in datasets["contradiction"] if str(r["id"]) in lookback_predictions],
    predictions=lookback_predictions,
    n=5,
)

show_prediction_examples(
    rows=[r for r in datasets["missing_tool"] if str(r["id"]) in lookback_predictions],
    predictions=lookback_predictions,
    n=3,
)

show_prediction_examples(
    rows=[r for r in datasets["overgeneration"] if str(r["id"]) in lookback_predictions],
    predictions=lookback_predictions,
    n=3,
)


Example 1
ID: 183_ep1_contradiction
TYPE: contradiction

QUERY:
Now, I'm considering a particular location for my next web scraping project. Could you give me information about the per capita income of the city with the geographical coordinates 34.0522° N, 118.2437° W?

CONTEXT:
[{"name": "Get City Income", "results": {"city_income": null, "city_name": null}}]

CLEAN OUTPUT (reference only, not used by detector):
It seems there was an issue retrieving the per capita income for the city with coordinates 34.0522° N, 118.2437° W. This could be due to various reasons:

1. **Data Availability**: The tool may not have data for this specific city or region, resulting in null values for `city_income` and `city_name`.

2. **Accuracy of Coordinates**: The provided coordinates might not precisely match a city recognized by the tool's database, leading to no results returned.

3. **Tool Limitations**: The tool itself might have limitations in terms of the cities or regions it can provide income da

## Improved method: contradiction-tuned Tool-aware DeBERTa

Enhancement over LettuceDetect / LookBackLens: fine-tune **microsoft/deberta-v3-base** as a token classifier with a **tool-aware context prompt** (query, available tools, tool call, tool output) + final answer, matching the core idea in `Updated_Hallucination_Detection_Pipeline.ipynb`.

- Training: `mixed_train`, with contradiction rows oversampled to target the weakest baseline slice.
- Model selection: after every epoch, search the threshold on `mixed_val` and save the checkpoint with the best validation **char F1**.
- Evaluation: one predict pass on `mixed_test` (same reporting as LettuceDetect / LookBackLens).
- Checkpoints: `data/checkpoints/toolaware_deberta_contradiction_tuned_model/`, predictions `toolaware_deberta_contradiction_tuned_test_predictions.jsonl`.

At inference time the detector sees only `query`, tool metadata, `tool_call`, `context`, and `output`. `clean_output` is printed for qualitative inspection only; `hallucination_labels` are used only for training/evaluation labels.

In [ ]:
%pip install -q transformers accelerate

In [21]:
import importlib

import toolaware_deberta_baseline as deberta_mod
importlib.reload(deberta_mod)

from toolaware_deberta_baseline import (
    ToolAwareDeBERTaConfig,
    ToolAwareDeBERTaDetector,
    default_model_dir,
    default_preds_path,
    predict_all as deberta_predict_all,
)
from lettuce_baseline import (
    CHECKPOINTS_DIR,
    evaluate_corruption_type_mix,
    evaluate_predictions,
    load_predictions,
    save_predictions,
    summarize_mixed_test,
)

assert "mixed_train" in globals() and "mixed_val" in globals() and "mixed_test" in globals(), (
    "Run the mixed-dataset cell first."
)

DEBERTA_RUN_NAME = "toolaware_deberta_contradiction_tuned"
DEBERTA_MODEL_DIR = default_model_dir(DEBERTA_RUN_NAME)
DEBERTA_PREDS_PATH = default_preds_path(DEBERTA_RUN_NAME)

FORCE_DEBERTA_TRAIN = False
FORCE_DEBERTA_PREDICT = False
DEBERTA_SMOKE = False  # True: 400 train / 100 val rows, 1 epoch

deberta_config = ToolAwareDeBERTaConfig(
    num_epochs=1 if DEBERTA_SMOKE else 8,
    batch_size=4,
    grad_accum_steps=2,
    contradiction_oversample=1 if DEBERTA_SMOKE else 3,
)

max_train_rows = 400 if DEBERTA_SMOKE else None
max_val_rows = 100 if DEBERTA_SMOKE else None
max_test_rows = 100 if DEBERTA_SMOKE else None

if FORCE_DEBERTA_TRAIN or not (DEBERTA_MODEL_DIR / "config.json").exists():
    deberta_detector = ToolAwareDeBERTaDetector(config=deberta_config)
    train_meta = deberta_detector.train(
        mixed_train,
        mixed_val,
        model_dir=DEBERTA_MODEL_DIR,
        max_train_rows=max_train_rows,
        max_val_rows=max_val_rows,
    )
    print(
        f"Trained DeBERTa: best_epoch={train_meta['best_epoch']}  "
        f"val_loss={train_meta['best_val_loss']:.4f}  "
        f"val_f1={train_meta['best_val_f1']:.4f}  "
        f"threshold={train_meta['best_threshold']:.2f}  "
        f"train_rows={train_meta['n_train']} -> {train_meta['n_train_after_oversample']}  "
        f"seconds={train_meta['training_seconds']:.0f}"
    )
    print(f"Saved model: {DEBERTA_MODEL_DIR.resolve()}")
else:
    deberta_detector = ToolAwareDeBERTaDetector.load(DEBERTA_MODEL_DIR)
    print(
        f"Loaded DeBERTa from {DEBERTA_MODEL_DIR.resolve()}  "
        f"(threshold={deberta_detector.threshold:.2f})"
    )

test_rows = mixed_test if max_test_rows is None else mixed_test[:max_test_rows]

if DEBERTA_PREDS_PATH.exists() and not FORCE_DEBERTA_PREDICT:
    deberta_predictions = load_predictions(DEBERTA_PREDS_PATH)
    print(f"Loaded DeBERTa predictions: {DEBERTA_PREDS_PATH.resolve()} ({len(deberta_predictions)} rows)")
else:
    deberta_predictions = deberta_predict_all(test_rows, deberta_detector)
    save_predictions(
        deberta_predictions,
        DEBERTA_PREDS_PATH,
        meta={
            "detector": "toolaware_deberta",
            "model_name": deberta_config.model_name,
            "split": "mixed_test",
            "n_rows": len(test_rows),
            "threshold": deberta_detector.threshold,
            "smoke": DEBERTA_SMOKE,
        },
    )
    print(f"Saved DeBERTa predictions: {DEBERTA_PREDS_PATH.resolve()} ({len(deberta_predictions)} rows)")

test_mix = summarize_mixed_test(test_rows)
print("\n=== Tool-aware DeBERTa on mixed_test (micro char P/R/F1) ===")
print(
    f"rows: {test_mix['n_total']} = corrupted {test_mix['n_corrupted']} + clean {test_mix['n_clean']}"
)
print(f"  corrupted by type: {test_mix['by_type']}")

deberta_mixed_result = evaluate_predictions(test_rows, deberta_predictions)
print(
    f"metrics: P={deberta_mixed_result['precision']:.4f}  "
    f"R={deberta_mixed_result['recall']:.4f}  "
    f"F1={deberta_mixed_result['f1']:.4f}"
)

print(
    "\nPer corruption type (corrupted rows of that type + all test clean; "
    "same predictions, no re-inference):"
)
for row in evaluate_corruption_type_mix(test_rows, deberta_predictions):
    print(
        f"  {row['hallucination_type']:22s}  "
        f"n={row['n']:3d} (corrupted={row['n_corrupted']:3d} + clean={row['n_clean']:3d})  "
        f"P={row['precision']:.4f}  R={row['recall']:.4f}  F1={row['f1']:.4f}"
    )

/home/slukina/miniconda3/envs/videoagent-siglip2/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoding train:   0%|          | 0/5427 [00:00<?, ?it/s]

Encoding validation:   0%|          | 0/514 [00:00<?, ?it/s]

DeBERTa epoch 1/8:   0%|          | 0/1357 [00:00<?, ?it/s]

/home/slukina/miniconda3/envs/videoagent-siglip2/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


...

Validation loss:   0%|          | 0/129 [00:00<?, ?it/s]

The tokenizer you are loading from 'data/checkpoints/toolaware_deberta_contradiction_tuned_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Epoch 8: train_loss=0.002311, val_loss=0.348465, val_f1=0.9427, threshold=0.15, contradiction_f1=0.6741
Trained DeBERTa: best_epoch=5  val_loss=0.2825  val_f1=0.9483  threshold=0.95  train_rows=3617 -> 5427  seconds=632
Saved model: /gpfs/home/slukina/personal/hallucination/data/checkpoints/toolaware_deberta_contradiction_tuned_model


Tool-aware DeBERTa:   0%|          | 0/1032 [00:00<?, ?row/s]

Saved DeBERTa predictions: /gpfs/home/slukina/personal/hallucination/data/checkpoints/toolaware_deberta_contradiction_tuned_test_predictions.jsonl (1032 rows)

=== Tool-aware DeBERTa on mixed_test (micro char P/R/F1) ===
rows: 1032 = corrupted 761 + clean 271
  corrupted by type: {'contradiction': 257, 'missing_tool': 233, 'overgeneration': 271}
metrics: P=0.9612  R=0.9286  F1=0.9446

Per corruption type (corrupted rows of that type + all test clean; same predictions, no re-inference):
  contradiction           n=528 (corrupted=257 + clean=271)  P=0.6884  R=0.6181  F1=0.6513
  missing_tool            n=504 (corrupted=233 + clean=271)  P=0.9655  R=0.9901  F1=0.9776
  overgeneration          n=542 (corrupted=271 + clean=271)  P=0.9582  R=0.9540  F1=0.9561


| Stage | Rows | What it is |
|---|---:|---|
| `mixed_train` (logical split) | **3617** | 942 clean + 2675 corrupted (`contradiction` 905 + `missing_tool` 828 + `overgeneration` 942) |
| After **contradiction oversampling ×3** | **5427** | Same 3617 base rows, plus **2 extra copies** of each **contradiction** train row only |
| `mixed_val` / `mixed_test` | 514 / 1032 | No oversampling |

```text
train_for_encoding = all mixed_train rows + (factor - 1) × contradiction rows
                   = 3617 + 2 × 905 = 5427
```

In [ ]:

show_prediction_examples(
    rows=[r for r in datasets["contradiction"] if str(r["id"]) in deberta_predictions],
    predictions=deberta_predictions,
    n=5,
)

show_prediction_examples(
    rows=[r for r in datasets["missing_tool"] if str(r["id"]) in deberta_predictions],
    predictions=deberta_predictions,
    n=3,
)

show_prediction_examples(
    rows=[r for r in datasets["overgeneration"] if str(r["id"]) in deberta_predictions],
    predictions=deberta_predictions,
    n=3,
)


Example 1
ID: 183_ep1_contradiction
TYPE: contradiction

QUERY:
Now, I'm considering a particular location for my next web scraping project. Could you give me information about the per capita income of the city with the geographical coordinates 34.0522° N, 118.2437° W?

CONTEXT:
[{"name": "Get City Income", "results": {"city_income": null, "city_name": null}}]

CLEAN OUTPUT (reference only, not used by detector):
It seems there was an issue retrieving the per capita income for the city with coordinates 34.0522° N, 118.2437° W. This could be due to various reasons:

1. **Data Availability**: The tool may not have data for this specific city or region, resulting in null values for `city_income` and `city_name`.

2. **Accuracy of Coordinates**: The provided coordinates might not precisely match a city recognized by the tool's database, leading to no results returned.

3. **Tool Limitations**: The tool itself might have limitations in terms of the cities or regions it can provide income da